# 07. Giải Thích Mô Hình (Explainable AI), Khung Lý Thuyết SEDL & Kiểm Định Công Bằng (Fairness Audit) Qua 4 Mốc Thời Gian
## Giai đoạn 8A (Tuần 8): Toàn cảnh XAI Đa Mốc (`25%`, `50%`, `75%`, `100%` Dynamic Active) & Đạo đức AI trên OULAD
**Dự án**: LMS Tiên tiến và Learning Analytics (Bộ dữ liệu OULAD)  
**Tác giả**: Vũ Hoà Bình (MSSV: 202416137)  

---

### Kiến trúc & Quy chuẩn Thực nghiệm XAI Đa Mốc (Zero Data Leakage trên 5-Fold Out-of-Fold):
1. **Đối tượng giải thích toàn diện qua cả 4 mốc tiến trình (`25%`, `50%`, `75%`, `100%` `dynamic_active`)**:
   - Lần lượt đánh giá trên 4 quần thể đang theo học thực tế: $N_{25\%} = 26,248$, $N_{50\%} = 24,601$, $N_{75\%} = 23,159$, $N_{100\%} = 22,522$ với đúng **114 đặc trưng đã mã hóa** và phân vùng **5-Fold Stratified Group CV** theo `id_student` cùng ngưỡng tối ưu nội bộ $\tau^*_k$ (từ Tuần 7).
   - Mốc `25%` nạp trực tiếp từ `models/best_model_5fold_ensemble.joblib` (`ROC-AUC = 0.8480`, `Recall = 0.7696`, `F1 = 0.7249`); các mốc `50%`, `75%`, `100%` tái lập chính xác quy trình Tuần 7 và đóng gói đồng bộ vào `models/best_model_5fold_4milestones.joblib` (giữ nguyên `best_model.pkl` và `best_model_5fold_ensemble.joblib` để tương thích ngược).
2. **Đầy đủ 5 kỹ thuật XAI bổ trợ lẫn nhau trên TẤT CẢ 4 mốc**:
   - **Built-in Feature Importance** (`Gain`, `Weight`, `Cover` trung bình 5 Fold) trên cả 4 mốc.
   - **5-Fold Out-of-Fold Permutation Importance** (`ROC-AUC` drop đo trên từng Test Fold — Zero Leakage, `n_repeats=5`) trên cả 4 mốc.
   - **Exact TreeSHAP** (tính chính xác 100% giá trị Shapley trong không gian log-odds bằng động cơ C++ `pred_contribs=True` của XGBoost cho toàn bộ mẫu OOF ở cả 4 mốc, sai số cộng tính $\le 10^{-6}$).
   - **Partial Dependence Plots (PDP & ICE) + SHAP Dependence** trên đơn vị gốc (Raw Units) cho 5 đặc trưng chủ chốt (`weighted_avg_score`, `submission_rate`, `days_since_last_activity`, `consistency_score`, `max_inactivity_streak`) qua cả 4 mốc `25% -> 50% -> 75% -> 100%`.
   - **Local Case Studies (`TP`, `FP`, `TN`, `FN` + Quỹ đạo Dọc `25%->100%`) & Local Surrogate LIME** (`LocalTabularSurrogateLIME`, 1,200 nhiễu loạn, `Ridge(alpha=1.0)` trên log-odds) trên **200 sinh viên OOF phân tầng mỗi mốc** ($\times 4$ mốc = 800 lượt kiểm định).
3. **Liên kết Khung Lý thuyết SEDL & Kiểm định Công bằng (Fairness Audit) Đa Mốc**:
   - Quy chiếu toàn bộ 114 đặc trưng về **4 chiều khung SEDL** (`Behavioural`, `Cognitive`, `Social`, `Demographic & Context`) và trực quan hóa Beeswarm + Tiến hóa SEDL từ `25%` đến `100%`.
   - Kiểm định công bằng trên 5 thuộc tính (`gender`, `disability`, `age_band`, `imd_band`, `region`) theo Equal Opportunity ($\Delta\text{TPR}$), Predictive Equality ($\Delta\text{FPR}$), $\Delta\text{AUC}$, Brier Score và Disparate Impact Ratio (quy tắc $4/5 \in [0.80, 1.25]$) xuyên suốt cả 4 mốc.

In [1]:
import json
import logging
import sys
import time
import unittest
import warnings
from pathlib import Path
from typing import Any, Dict, List, Tuple

import joblib
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import xgboost as xgb
from IPython.display import display
from scipy import stats
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import OneHotEncoder, RobustScaler

warnings.filterwarnings("ignore")

if sys.platform == "win32":
    try:
        sys.stdout.reconfigure(encoding="utf-8")
        sys.stderr.reconfigure(encoding="utf-8")
    except Exception:
        pass

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)
logger = logging.getLogger("OULAD_Week8_XAI")

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["font.sans-serif"] = "DejaVu Sans"
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: "%.4f" % x)

# Kiểm tra gói shap (nếu Windows Application Control chặn _cext thì kích hoạt XGBoost Native C++ Exact TreeSHAP)
SHAP_PKG_AVAILABLE = False
SHAP_STATUS_MSG = ""
try:
    import shap  # type: ignore
    _dummy_m = xgb.XGBClassifier(n_estimators=2, max_depth=2, random_state=42)
    _dummy_m.fit(np.array([[0.0, 1.0], [1.0, 0.0], [0.2, 0.8], [0.8, 0.2]]), [0, 1, 0, 1])
    _ = shap.TreeExplainer(_dummy_m).shap_values(np.array([[0.1, 0.9]]))
    SHAP_PKG_AVAILABLE = True
    SHAP_STATUS_MSG = f"shap=={getattr(shap, '__version__', 'installed')} hoạt động bình thường."
except Exception as exc:
    SHAP_PKG_AVAILABLE = False
    SHAP_STATUS_MSG = (
        f"shap C-extension bị chặn/không khả dụng ({type(exc).__name__}); "
        "kích hoạt XGBoost Native C++ Exact TreeSHAP (pred_contribs=True, Lundberg et al., 2020)."
    )

LIME_PKG_AVAILABLE = False
try:
    import lime  # type: ignore
    LIME_PKG_AVAILABLE = True
except Exception:
    LIME_PKG_AVAILABLE = False

PROJECT_ROOT = Path("..").resolve() if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
SNAPSHOTS_DIR = PROCESSED_DIR / "week5_6_snapshots"
WEEK6_DIR = PROCESSED_DIR / "week6_advanced_features"
WEEK7_DIR = PROCESSED_DIR / "week7_modeling"
WEEK8_XAI_DIR = PROCESSED_DIR / "week8_xai"
REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"
MODELS_DIR = PROJECT_ROOT / "models"

for d in [WEEK8_XAI_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

MILESTONES = ["25%", "50%", "75%", "100%"]
MILESTONE_COLORS = {
    "25%": "#2980b9",
    "50%": "#16a085",
    "75%": "#e67e22",
    "100%": "#c0392b",
}

print(f"Project Root   : {PROJECT_ROOT}")
print(f"Week 8 XAI Dir : {WEEK8_XAI_DIR}")
print(f"Figures Dir    : {FIGURES_DIR}")
print(f"SHAP Engine    : {SHAP_STATUS_MSG}")
print(f"LIME Engine    : {'Installed' if LIME_PKG_AVAILABLE else 'Self-contained LocalTabularSurrogateLIME (Ribeiro et al., 2016)'}")

Project Root   : C:\Users\mrore\Project_I
Week 8 XAI Dir : C:\Users\mrore\Project_I\data\processed\week8_xai
Figures Dir    : C:\Users\mrore\Project_I\reports\figures
SHAP Engine    : shap==0.53.0 hoạt động bình thường.
LIME Engine    : Self-contained LocalTabularSurrogateLIME (Ribeiro et al., 2016)


## 1. Tích hợp `OULAD5FoldCVEngine`, Ánh xạ 4 Chiều SEDL & Kiểm chứng OOF + Exact TreeSHAP qua 4 Mốc (`25%`, `50%`, `75%`, `100%`)
Tái định nghĩa `OULAD5FoldCVEngine` chuẩn Tuần 7 (Zero Data Leakage), nạp mô hình `25%` từ `models/best_model_5fold_ensemble.joblib` và tái lập chính xác In-Fold training + In-Fold threshold tuning ($\tau^*_k$) cho `50%`, `75%`, `100%` khớp 100% kết quả Tuần 7 (`week7_experiment_metrics.json`), kiểm chứng sai số cộng tính Exact TreeSHAP $\le 10^{-6}$ trên toàn bộ 20 Folds (5 Fold $\times$ 4 Mốc), rồi lưu gói 4 mốc vào `models/best_model_5fold_4milestones.joblib`.

In [2]:
class OULAD5FoldCVEngine:
    '''Động cơ 5-Fold Stratified Group CV và tiền xử lý In-Fold Zero Data Leakage (chuẩn Tuần 7).'''

    ALL_ACTIVITY_TYPES = [
        "dataplus", "dualpane", "externalquiz", "folder", "forumng",
        "glossary", "homepage", "htmlactivity", "oucollaborate", "oucontent",
        "ouelluminate", "ouwiki", "page", "questionnaire", "quiz",
        "repeatactivity", "resource", "sharedsubpage", "subpage", "url",
    ]

    ORDINAL_MAPPINGS = {
        "highest_education": {
            "No Formal quals": 0,
            "Lower Than A Level": 1,
            "A Level or Equivalent": 2,
            "HE Qualification": 3,
            "Post Graduate Qualification": 4,
        },
        "imd_band": {
            "0-10%": 0, "10-20": 1, "20-30%": 2, "30-40%": 3, "40-50%": 4,
            "50-60%": 5, "60-70%": 6, "70-80%": 7, "80-90%": 8, "90-100%": 9,
            "Unknown": 4,
        },
        "age_band": {"0-35": 0, "35-55": 1, "55<=": 2},
    }

    NOMINAL_COLS = ["gender", "disability", "region", "code_module"]
    ORDINAL_COLS = ["highest_education", "imd_band", "age_band"]
    FLAG_COLS = ["imd_was_missing", "has_no_submissions", "has_no_clicks"]

    @classmethod
    def get_week6_numeric_cols(cls) -> List[str]:
        all_20_act_cols = [f"clicks_{act}" for act in cls.ALL_ACTIVITY_TYPES]
        return [
            "total_clicks", "active_days", "clicks_per_day", "clicks_per_active_day",
            "click_std", "click_cv", "active_days_in_course", "active_days_ratio",
            "pre_course_clicks", "has_pre_course_clicks", "active_span_days",
            *all_20_act_cols,
            "cognitive_clicks", "social_clicks", "oucontent_ratio", "forumng_ratio",
            "quiz_ratio", "resource_ratio", "glossary_ratio", "cognitive_ratio",
            "social_ratio", "activity_diversity", "activity_entropy",
            "total_course_assessments", "due_course_assessments", "avg_score",
            "weighted_avg_score", "earned_weight_score", "min_score", "max_score",
            "score_std", "first_score", "num_submissions", "num_submissions_raw",
            "submission_rate", "course_progress_submission_rate", "avg_days_before_deadline",
            "late_submissions", "late_submission_rate", "tma1_score", "tma1_days_early",
            "submitted_tma1",
            "num_of_prev_attempts", "studied_credits", "module_presentation_length",
            "days_before_start", "is_repeat_student", "credit_overload", "is_late_registration",
            "clicks_week_1_to_4", "clicks_last_2_weeks", "clicks_trend", "engagement_decay",
            "days_since_last_activity", "inactivity_ratio", "max_inactivity_streak", "consistency_score",
            "active_weeks", "login_regularity", "weekend_clicks", "weekend_ratio",
            "burst_count", "burst_ratio",
            "last_minus_first_score", "score_trend_slope",
        ]

    @staticmethod
    def filter_cohort(df: pd.DataFrame, cohort_type: str = "dynamic_active") -> pd.DataFrame:
        if cohort_type == "full":
            return df.copy().reset_index(drop=True)
        elif cohort_type == "dynamic_active":
            mask = df["date_unregistration"].isna() | (df["date_unregistration"] > df["cutoff_day"])
            return df[mask].copy().reset_index(drop=True)
        raise ValueError(f"cohort_type không hợp lệ: {cohort_type}")

    @staticmethod
    def get_5fold_splits(
        df: pd.DataFrame, n_splits: int = 5, random_state: int = 42
    ) -> List[Tuple[int, np.ndarray, np.ndarray]]:
        sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
        y = df["at_risk"].values
        groups = df["id_student"].values
        return [(fold, tr_idx, te_idx) for fold, (tr_idx, te_idx) in enumerate(sgkf.split(df, y, groups=groups), start=1)]

    def fit_preprocessor(self, train_df: pd.DataFrame) -> Dict[str, Any]:
        num_cols = self.get_week6_numeric_cols()
        imd_mode = train_df["imd_band"].mode().iloc[0] if not train_df["imd_band"].mode().empty else "20-30%"
        reg_median = float(train_df["date_registration"].median())
        skewed_cols = [
            c for c in num_cols
            if (
                (c.startswith("clicks_") and c != "clicks_trend")
                or c in [
                    "total_clicks", "active_days", "active_days_in_course",
                    "daily_click_mean", "daily_click_std", "click_std",
                    "pre_course_clicks", "cognitive_clicks", "social_clicks",
                    "weekend_clicks", "studied_credits",
                ]
            )
        ]
        bounds: Dict[str, Tuple[float, float]] = {}
        for col in skewed_cols:
            s = train_df[col].dropna()
            q25, q75 = float(s.quantile(0.25)), float(s.quantile(0.75))
            iqr = q75 - q25
            if iqr > 0:
                bounds[col] = (0.0, q75 + 3.0 * iqr)

        ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
        ohe.fit(train_df[self.NOMINAL_COLS].fillna("Missing").astype(str))
        ohe_names = list(ohe.get_feature_names_out(self.NOMINAL_COLS))

        temp_state = {
            "num_cols": num_cols,
            "imd_mode": imd_mode,
            "reg_median": reg_median,
            "skewed_cols": skewed_cols,
            "bounds": bounds,
            "ohe": ohe,
            "ohe_names": ohe_names,
            "cols_to_scale": num_cols + self.ORDINAL_COLS,
            "scaler": None,
        }
        tr_clean = self._clean_df(train_df, temp_state)
        scaler = RobustScaler()
        scaler.fit(tr_clean[temp_state["cols_to_scale"]])
        temp_state["scaler"] = scaler
        return temp_state

    def _clean_df(self, df_in: pd.DataFrame, state: Dict[str, Any]) -> pd.DataFrame:
        d = df_in.copy()
        d["imd_was_missing"] = d["imd_band"].isna().astype(float)
        d["has_no_submissions"] = (d["num_submissions"].fillna(0) == 0).astype(float)
        d["has_no_clicks"] = (d["total_clicks"].fillna(0) == 0).astype(float)
        d["imd_band"] = d["imd_band"].fillna(state["imd_mode"])
        d["date_registration"] = d["date_registration"].fillna(state["reg_median"])
        d["days_before_start"] = -d["date_registration"]

        for sc in [
            "avg_score", "weighted_avg_score", "earned_weight_score",
            "min_score", "max_score", "score_std", "first_score", "tma1_score",
        ]:
            if sc in d.columns:
                d[sc] = d[sc].fillna(0.0)
        for ec in ["avg_days_early", "avg_days_before_deadline", "tma1_days_early"]:
            if ec in d.columns:
                d[ec] = d[ec].fillna(-10.0)
        for c in state["num_cols"]:
            if c in d.columns:
                d[c] = d[c].fillna(0.0)
        for col, mapping in self.ORDINAL_MAPPINGS.items():
            d[col] = d[col].map(mapping).fillna(mapping.get("Unknown", 2)).astype(float)
        for col, (low, high) in state["bounds"].items():
            if col in d.columns:
                d[col] = d[col].clip(lower=low, upper=high)
        for col in state["skewed_cols"]:
            if col in d.columns and "click" in col and "ratio" not in col and "per" not in col and "cv" not in col:
                d[col] = np.log1p(np.maximum(0.0, d[col]))
        return d

    def transform_with_state(self, df_in: pd.DataFrame, state: Dict[str, Any]) -> pd.DataFrame:
        d_clean = self._clean_df(df_in, state)
        cols_to_scale = state["cols_to_scale"]
        scaled_arr = state["scaler"].transform(d_clean[cols_to_scale])
        scaled_df = pd.DataFrame(scaled_arr, columns=cols_to_scale, index=df_in.index)
        ohe_arr = state["ohe"].transform(df_in[self.NOMINAL_COLS].fillna("Missing").astype(str))
        ohe_df = pd.DataFrame(ohe_arr, columns=state["ohe_names"], index=df_in.index)
        flags_df = d_clean[self.FLAG_COLS].copy()
        out = pd.concat([scaled_df, ohe_df, flags_df], axis=1)
        return out.replace([np.inf, -np.inf], 0.0).fillna(0.0)

    def prepare_fold_matrices(
        self, train_df: pd.DataFrame, test_df: pd.DataFrame
    ) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, Any]]:
        state = self.fit_preprocessor(train_df)
        return (
            self.transform_with_state(train_df, state),
            self.transform_with_state(test_df, state),
            state,
        )


def tune_threshold_inner_val(
    y_val: np.ndarray, y_prob: np.ndarray, min_recall_target: float = 0.76
) -> Tuple[float, float, float]:
    '''Tìm ngưỡng tau* tối ưu F1 trên Inner-Val sao cho Recall >= min_recall_target (chuẩn Tuần 7).'''
    best_tau, best_f1, best_rec = 0.50, -1.0, 0.0
    for tau in np.arange(0.15, 0.71, 0.01):
        pred = (y_prob >= tau).astype(int)
        rec = float(recall_score(y_val, pred, zero_division=0))
        f1 = float(f1_score(y_val, pred, zero_division=0))
        if rec >= min_recall_target and f1 > best_f1:
            best_tau = round(float(tau), 2)
            best_f1 = f1
            best_rec = rec
    if best_f1 < 0:
        for tau in np.arange(0.15, 0.71, 0.01):
            pred = (y_prob >= tau).astype(int)
            rec = float(recall_score(y_val, pred, zero_division=0))
            f1 = float(f1_score(y_val, pred, zero_division=0))
            if (rec > best_rec) or (abs(rec - best_rec) < 1e-6 and f1 > best_f1):
                best_tau = round(float(tau), 2)
                best_f1 = f1
                best_rec = rec
    return best_tau, round(best_rec, 4), round(best_f1, 4)


def compute_exact_treeshap(
    model: xgb.XGBClassifier, X_df: pd.DataFrame, feature_names: List[str]
) -> Tuple[np.ndarray, np.ndarray]:
    '''Tính Exact TreeSHAP trong không gian log-odds qua XGBoost native C++ pred_contribs=True.'''
    dm = xgb.DMatrix(X_df[feature_names], feature_names=feature_names)
    contribs = model.get_booster().predict(dm, pred_contribs=True)
    return contribs[:, :-1].astype(float), contribs[:, -1].astype(float)


# Nạp artifacts Tuần 7 & 4 snapshots
engine = OULAD5FoldCVEngine()
best_model_pkl = joblib.load(MODELS_DIR / "best_model.pkl")
ensemble_bundle = joblib.load(MODELS_DIR / "best_model_5fold_ensemble.joblib")
with open(MODELS_DIR / "best_model_metadata.json", "r", encoding="utf-8") as f:
    best_metadata = json.load(f)
with open(WEEK7_DIR / "week7_experiment_metrics.json", "r", encoding="utf-8") as f:
    week7_metrics = json.load(f)

FEAT_NAMES_114: List[str] = ensemble_bundle["feature_names_114"]
CONSENSUS_PARAMS: Dict[str, Any] = best_metadata["consensus_params"]
FOLD_THRESHOLDS_25: List[float] = [float(x) for x in ensemble_bundle["fold_thresholds"]]

snapshots = {
    "25%": pd.read_csv(SNAPSHOTS_DIR / "features_snapshot_25pct.csv"),
    "50%": pd.read_csv(SNAPSHOTS_DIR / "features_snapshot_50pct.csv"),
    "75%": pd.read_csv(SNAPSHOTS_DIR / "features_snapshot_75pct.csv"),
    "100%": pd.read_csv(SNAPSHOTS_DIR / "features_snapshot_100pct.csv"),
}
catalog_w6 = pd.read_csv(WEEK6_DIR / "feature_catalog_week6.csv")
RAW_SEDL_MAP = dict(zip(catalog_w6["feature_name"], catalog_w6["sedl_dimension"]))
RAW_DESC_MAP = dict(zip(catalog_w6["feature_name"], catalog_w6["description"]))

def map_to_sedl_4d(feat_name: str) -> str:
    if feat_name in ("has_no_submissions", "has_no_clicks"):
        return "Behavioural"
    if feat_name == "imd_was_missing" or feat_name.startswith(("gender_", "disability_", "region_", "code_module_")):
        return "Demographic & Context"
    raw_dim = RAW_SEDL_MAP.get(feat_name, "")
    if raw_dim.startswith("Behavioural") or raw_dim == "Emotional / Feedback (Proxy)":
        return "Behavioural"
    if raw_dim.startswith("Cognitive"):
        return "Cognitive"
    if raw_dim.startswith("Social"):
        return "Social"
    if raw_dim in (
        "Demographic", "Demographic (Prior Knowledge)", "Demographic (Socio-Economic)",
        "Academic History", "Academic Load", "Course Context", "Context (Course Structure)",
    ):
        return "Demographic & Context"
    raise ValueError(f"Chưa ánh xạ SEDL cho biến: {feat_name} ({raw_dim})")

SEDL_4D_MAP: Dict[str, str] = {fn: map_to_sedl_4d(fn) for fn in FEAT_NAMES_114}
SEDL_DIM_ORDER = ["Behavioural", "Cognitive", "Social", "Demographic & Context"]
SEDL_COLORS = {
    "Behavioural": "#2980b9",
    "Cognitive": "#8e44ad",
    "Social": "#16a085",
    "Demographic & Context": "#d35400",
}

def describe_feature_vi(fn: str) -> str:
    if fn in RAW_DESC_MAP:
        return str(RAW_DESC_MAP[fn])
    if fn == "imd_was_missing":
        return "Cờ thiếu thông tin chỉ số IMD gốc"
    if fn == "has_no_submissions":
        return "Cờ chưa có bài tập nộp đến mốc cắt"
    if fn == "has_no_clicks":
        return "Cờ chưa có lượt tương tác VLE đến mốc cắt"
    if fn.startswith("code_module_"):
        return f"Thuộc học phần {fn.replace('code_module_', '')}"
    if fn.startswith("region_"):
        return f"Cư trú tại khu vực {fn.replace('region_', '')}"
    if fn.startswith("gender_"):
        return f"Giới tính {fn.replace('gender_', '')}"
    if fn.startswith("disability_"):
        return f"Tình trạng khuyết tật ({fn.replace('disability_', '')})"
    return fn

print("Phân bố 114 đặc trưng theo 4 chiều SEDL:", pd.Series(SEDL_4D_MAP).value_counts().to_dict())

# =========================================================================
# Tái lập & Kiểm chứng 5-Fold OOF + Exact TreeSHAP cho TẤT CẢ 4 MỐC (25%, 50%, 75%, 100%)
# =========================================================================
t_sec1 = time.time()
ms_results: Dict[str, Dict[str, Any]] = {}
bundle_4ms_milestones: Dict[str, Dict[str, Any]] = {}
perf_summary_rows: List[Dict[str, Any]] = []
global_max_shap_err = 0.0

w7_early_lookup = {
    (r["snapshot"], r["cohort"], r["model"]): r
    for r in week7_metrics["early_prediction_summary"]
}

for snap_label in MILESTONES:
    df_m = engine.filter_cohort(snapshots[snap_label], "dynamic_active")
    y_m = df_m["at_risk"].values.astype(int)
    g_m = df_m["id_student"].values
    splits_m = engine.get_5fold_splits(df_m, n_splits=5, random_state=42)

    oof_probs_m = np.zeros(len(df_m), dtype=float)
    oof_preds_m = np.zeros(len(df_m), dtype=int)
    oof_folds_m = np.zeros(len(df_m), dtype=int)
    oof_taus_m = np.zeros(len(df_m), dtype=float)
    oof_shap_m = np.zeros((len(df_m), len(FEAT_NAMES_114)), dtype=float)
    oof_base_m = np.zeros(len(df_m), dtype=float)
    X_oof_scaled_m = pd.DataFrame(0.0, index=df_m.index, columns=FEAT_NAMES_114)

    if snap_label == "25%":
        fold_models_m = list(ensemble_bundle["fold_models"])
        fold_preprocessors_m = list(ensemble_bundle["fold_preprocessors"])
        fold_taus_m = list(FOLD_THRESHOLDS_25)
        full_state_m = ensemble_bundle["full_preprocessor_state"]
        full_model_m = best_model_pkl
    else:
        fold_models_m = []
        fold_preprocessors_m = []
        fold_taus_m = []
        for fold, tr_idx, te_idx in splits_m:
            tr_df_f = df_m.iloc[tr_idx].reset_index(drop=True)
            te_df_f = df_m.iloc[te_idx].reset_index(drop=True)
            X_tr_f, _, state_k = engine.prepare_fold_matrices(tr_df_f, te_df_f)
            X_tr_f = X_tr_f[FEAT_NAMES_114]
            y_tr_f = y_m[tr_idx]
            g_tr_f = g_m[tr_idx]
            pos_w = float((y_tr_f == 0).sum() / max(1, (y_tr_f == 1).sum()))

            in_sgkf = StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=200 + fold)
            in_tr_idx, in_va_idx = next(in_sgkf.split(tr_df_f, y_tr_f, groups=g_tr_f))
            X_in_tr, X_in_va, _ = engine.prepare_fold_matrices(tr_df_f.iloc[in_tr_idx], tr_df_f.iloc[in_va_idx])
            X_in_tr, X_in_va = X_in_tr[FEAT_NAMES_114], X_in_va[FEAT_NAMES_114]
            y_in_tr, y_in_va = y_tr_f[in_tr_idx], y_tr_f[in_va_idx]
            pos_w_in = float((y_in_tr == 0).sum() / max(1, (y_in_tr == 1).sum()))

            clf_in = xgb.XGBClassifier(
                scale_pos_weight=pos_w_in, eval_metric="logloss", random_state=42, n_jobs=-1, **CONSENSUS_PARAMS
            )
            clf_in.fit(X_in_tr, y_in_tr)
            tau_k, _, _ = tune_threshold_inner_val(y_in_va, clf_in.predict_proba(X_in_va)[:, 1], min_recall_target=0.76)

            clf_k = xgb.XGBClassifier(
                scale_pos_weight=pos_w, eval_metric="logloss", random_state=42, n_jobs=-1, **CONSENSUS_PARAMS
            )
            clf_k.fit(X_tr_f, y_tr_f)

            fold_models_m.append(clf_k)
            fold_preprocessors_m.append(state_k)
            fold_taus_m.append(float(tau_k))

        full_state_m = engine.fit_preprocessor(df_m)
        X_full_m = engine.transform_with_state(df_m, full_state_m)[FEAT_NAMES_114]
        pos_w_full = float((y_m == 0).sum() / max(1, (y_m == 1).sum()))
        full_model_m = xgb.XGBClassifier(
            scale_pos_weight=pos_w_full, eval_metric="logloss", random_state=42, n_jobs=-1, **CONSENSUS_PARAMS
        )
        full_model_m.fit(X_full_m, y_m)

    fold_metrics_m = []
    max_err_m = 0.0
    for fold, tr_idx, te_idx in splits_m:
        state_k = fold_preprocessors_m[fold - 1]
        model_k = fold_models_m[fold - 1]
        tau_k = float(fold_taus_m[fold - 1])
        X_te = engine.transform_with_state(df_m.iloc[te_idx], state_k)[FEAT_NAMES_114]
        X_oof_scaled_m.iloc[te_idx] = X_te.values

        probs = model_k.predict_proba(X_te)[:, 1]
        preds = (probs >= tau_k).astype(int)
        oof_probs_m[te_idx] = probs
        oof_preds_m[te_idx] = preds
        oof_folds_m[te_idx] = fold
        oof_taus_m[te_idx] = tau_k

        sv, bv = compute_exact_treeshap(model_k, X_te, FEAT_NAMES_114)
        oof_shap_m[te_idx] = sv
        oof_base_m[te_idx] = bv

        prob_recon = 1.0 / (1.0 + np.exp(-(bv + sv.sum(axis=1))))
        fold_err = float(np.max(np.abs(prob_recon - probs)))
        max_err_m = max(max_err_m, fold_err)
        global_max_shap_err = max(global_max_shap_err, fold_err)

        fold_metrics_m.append({
            "fold": int(fold),
            "n_test": int(len(te_idx)),
            "tau_k": float(tau_k),
            "roc_auc": float(roc_auc_score(y_m[te_idx], probs)),
            "pr_auc": float(average_precision_score(y_m[te_idx], probs)),
            "recall": float(recall_score(y_m[te_idx], preds)),
            "precision": float(precision_score(y_m[te_idx], preds)),
            "f1": float(f1_score(y_m[te_idx], preds)),
            "accuracy": float(accuracy_score(y_m[te_idx], preds)),
            "brier": float(brier_score_loss(y_m[te_idx], probs)),
            "max_shap_err": fold_err,
        })

    fdf_m = pd.DataFrame(fold_metrics_m)
    tn_m, fp_m, fn_m, tp_m = confusion_matrix(y_m, oof_preds_m, labels=[0, 1]).ravel()
    w7_ref = w7_early_lookup[(snap_label, "dynamic_active", "XGBoost (Tuned)")]

    perf_summary_rows.append({
        "milestone": snap_label,
        "n_cohort": int(len(df_m)),
        "at_risk_rate": round(float(y_m.mean()), 4),
        "tau_mean": round(float(np.mean(fold_taus_m)), 4),
        "roc_auc_mean": round(float(fdf_m["roc_auc"].mean()), 4),
        "w7_roc_auc": w7_ref["roc_auc_mean"],
        "pr_auc_mean": round(float(fdf_m["pr_auc"].mean()), 4),
        "recall_mean": round(float(fdf_m["recall"].mean()), 4),
        "w7_recall": w7_ref["recall_mean"],
        "precision_mean": round(float(fdf_m["precision"].mean()), 4),
        "f1_mean": round(float(fdf_m["f1"].mean()), 4),
        "w7_f1": w7_ref["f1_mean"],
        "accuracy_mean": round(float(fdf_m["accuracy"].mean()), 4),
        "brier_mean": round(float(fdf_m["brier"].mean()), 4),
        "max_shap_err": max_err_m,
        "tn": int(tn_m), "fp": int(fp_m), "fn": int(fn_m), "tp": int(tp_m),
    })

    ms_results[snap_label] = {
        "df": df_m,
        "y": y_m,
        "splits": splits_m,
        "oof_probs": oof_probs_m,
        "oof_preds": oof_preds_m,
        "oof_folds": oof_folds_m,
        "oof_taus": oof_taus_m,
        "oof_shap": oof_shap_m,
        "oof_base": oof_base_m,
        "X_oof_scaled": X_oof_scaled_m,
        "fold_models": fold_models_m,
        "fold_preprocessors": fold_preprocessors_m,
        "fold_thresholds": fold_taus_m,
        "full_state": full_state_m,
        "full_model": full_model_m,
        "fold_metrics": fold_metrics_m,
        "fold_df": fdf_m,
        "max_shap_err": max_err_m,
        "oof_cm": {"tn": int(tn_m), "fp": int(fp_m), "fn": int(fn_m), "tp": int(tp_m)},
    }

    bundle_4ms_milestones[snap_label] = {
        "n_cohort": int(len(df_m)),
        "at_risk_rate": round(float(y_m.mean()), 4),
        "fold_models": fold_models_m,
        "fold_preprocessors": fold_preprocessors_m,
        "fold_thresholds": fold_taus_m,
        "mean_threshold": float(np.mean(fold_taus_m)),
        "full_model": full_model_m,
        "full_preprocessor_state": full_state_m,
    }

# Lưu gói ensemble 4 mốc vào models/best_model_5fold_4milestones.joblib (giữ nguyên best_model.pkl & best_model_5fold_ensemble.joblib)
bundle_4ms_path = MODELS_DIR / "best_model_5fold_4milestones.joblib"
joblib.dump(
    {
        "model_name": best_metadata["best_tree_model_saved"],
        "feature_names_114": FEAT_NAMES_114,
        "consensus_params": CONSENSUS_PARAMS,
        "milestones": bundle_4ms_milestones,
    },
    bundle_4ms_path,
    compress=3,
)

# Gán biến alias mốc 25% để tương thích các kiểm chứng chi tiết
df_25 = ms_results["25%"]["df"]
y_25 = ms_results["25%"]["y"]
splits_25 = ms_results["25%"]["splits"]
oof_probs = ms_results["25%"]["oof_probs"]
oof_preds = ms_results["25%"]["oof_preds"]
oof_folds = ms_results["25%"]["oof_folds"]
oof_taus = ms_results["25%"]["oof_taus"]
oof_shap_25 = ms_results["25%"]["oof_shap"]
oof_base_25 = ms_results["25%"]["oof_base"]
X_oof_scaled = ms_results["25%"]["X_oof_scaled"]
fold_check_df = ms_results["25%"]["fold_df"]
fold_metrics_check = ms_results["25%"]["fold_metrics"]
max_additivity_err = ms_results["25%"]["max_shap_err"]

perf_summary_df = pd.DataFrame(perf_summary_rows)
logger.info(f"Hoàn tất 5-Fold OOF & Exact TreeSHAP qua 4 mốc trong {time.time() - t_sec1:.1f}s | Global Max SHAP Err={global_max_shap_err:.2e}")
print("=== ĐỐI KHỚP 5-FOLD OOF & EXACT TREESHAP QUA 4 MỐC (25% -> 100%) VS TUẦN 7 ===")
display(perf_summary_df[["milestone", "n_cohort", "tau_mean", "roc_auc_mean", "w7_roc_auc", "recall_mean", "w7_recall", "f1_mean", "w7_f1", "brier_mean", "max_shap_err"]])
print(f"Đã lưu gói mô hình 4 mốc tại: {bundle_4ms_path}")

Phân bố 114 đặc trưng theo 4 chiều SEDL: {'Behavioural': 42, 'Demographic & Context': 35, 'Cognitive': 29, 'Social': 8}


2026-11-25 21:36:55,089 [INFO] Hoàn tất 5-Fold OOF & Exact TreeSHAP qua 4 mốc trong 178.1s | Global Max SHAP Err=4.42e-07


=== ĐỐI KHỚP 5-FOLD OOF & EXACT TREESHAP QUA 4 MỐC (25% -> 100%) VS TUẦN 7 ===


,milestone,n_cohort,tau_mean,roc_auc_mean,w7_roc_auc,recall_mean,w7_recall,f1_mean,w7_f1,brier_mean,max_shap_err
0,25%,26248,0.4320,0.8480,0.8480,0.7696,0.7696,0.7249,0.7249,0.1544,0.0000
1,50%,24601,0.4960,0.9068,0.9068,0.7813,0.7812,0.7808,0.7808,0.1158,0.0000
2,75%,23159,0.6200,0.9457,0.9457,0.7817,0.7817,0.8299,0.8299,0.0839,0.0000
3,100%,22522,0.5740,0.9655,0.9655,0.8334,0.8334,0.8645,0.8645,0.0633,0.0000


Đã lưu gói mô hình 4 mốc tại: C:\Users\mrore\Project_I\models\best_model_5fold_4milestones.joblib


## 2. XAI Toàn cục (Global Explainability) qua 4 Mốc: Đối sánh 3 Phương pháp & Top 10 Đồng thuận SEDL
Tính đồng bộ trên cả **4 mốc (`25%`, `50%`, `75%`, `100%` Dynamic Active)** đối với toàn bộ **114 đặc trưng**:
1. **XGBoost Built-in Gain / Weight / Cover** (trung bình 5 Fold).
2. **5-Fold Out-of-Fold Permutation Importance** ($\Delta\text{ROC-AUC}$ trên từng Test Fold $k$, `n_repeats=5` — Zero Leakage).
3. **Exact TreeSHAP Global Importance** ($\mathbb{E}[|\phi_j|]$ trên $100\%$ mẫu OOF).  
Xuất **Hình 38** (đối sánh 3 phương pháp qua 4 mốc, `2×2`) và **Hình 39** (Top 10 đồng thuận SEDL qua 4 mốc, `2×2`).

In [3]:
t_global = time.time()

def norm_share_pct(arr: np.ndarray) -> np.ndarray:
    pos = np.maximum(0.0, arr)
    s = pos.sum()
    return 100.0 * pos / s if s > 0 else pos

consensus_by_ms: Dict[str, pd.DataFrame] = {}
global_rank_agreement_4ms: Dict[str, Dict[str, float]] = {}

for snap_label in MILESTONES:
    res_m = ms_results[snap_label]
    df_m = res_m["df"]
    y_m = res_m["y"]
    splits_m = res_m["splits"]
    X_oof_m = res_m["X_oof_scaled"]
    oof_shap_m = res_m["oof_shap"]
    fold_models_m = res_m["fold_models"]

    gain_mat = np.zeros((5, len(FEAT_NAMES_114)), dtype=float)
    weight_mat = np.zeros((5, len(FEAT_NAMES_114)), dtype=float)
    cover_mat = np.zeros((5, len(FEAT_NAMES_114)), dtype=float)
    perm_mat = np.zeros((5, len(FEAT_NAMES_114)), dtype=float)

    for fold, _, te_idx in splits_m:
        model_k = fold_models_m[fold - 1]
        booster = model_k.get_booster()
        g_dict = booster.get_score(importance_type="gain")
        w_dict = booster.get_score(importance_type="weight")
        c_dict = booster.get_score(importance_type="cover")
        gain_mat[fold - 1] = [g_dict.get(fn, 0.0) for fn in FEAT_NAMES_114]
        weight_mat[fold - 1] = [w_dict.get(fn, 0.0) for fn in FEAT_NAMES_114]
        cover_mat[fold - 1] = [c_dict.get(fn, 0.0) for fn in FEAT_NAMES_114]

        X_te = X_oof_m.iloc[te_idx]
        y_te = y_m[te_idx]
        pi = permutation_importance(
            model_k, X_te, y_te, scoring="roc_auc", n_repeats=5, random_state=42 + fold, n_jobs=-1
        )
        perm_mat[fold - 1] = pi.importances_mean

    mean_abs_shap = np.mean(np.abs(oof_shap_m), axis=0)
    mean_gain = gain_mat.mean(axis=0)
    mean_weight = weight_mat.mean(axis=0)
    mean_cover = cover_mat.mean(axis=0)
    mean_perm = perm_mat.mean(axis=0)
    sd_perm = perm_mat.std(axis=0, ddof=1)

    shap_share = norm_share_pct(mean_abs_shap)
    perm_share = norm_share_pct(mean_perm)
    gain_share = norm_share_pct(mean_gain)

    c_df = pd.DataFrame({
        "milestone": snap_label,
        "feature_name": FEAT_NAMES_114,
        "sedl_dimension": [SEDL_4D_MAP[fn] for fn in FEAT_NAMES_114],
        "description_vi": [describe_feature_vi(fn) for fn in FEAT_NAMES_114],
        "shap_mean_abs": mean_abs_shap,
        "shap_share_pct": shap_share,
        "perm_auc_drop_mean": mean_perm,
        "perm_auc_drop_sd": sd_perm,
        "perm_share_pct": perm_share,
        "xgb_gain_mean": mean_gain,
        "xgb_gain_share_pct": gain_share,
        "xgb_weight_mean": mean_weight,
        "xgb_cover_mean": mean_cover,
    })
    c_df["rank_shap"] = c_df["shap_mean_abs"].rank(ascending=False, method="min").astype(int)
    c_df["rank_perm"] = c_df["perm_auc_drop_mean"].rank(ascending=False, method="min").astype(int)
    c_df["rank_gain"] = c_df["xgb_gain_mean"].rank(ascending=False, method="min").astype(int)
    c_df["consensus_score_pct"] = (
        0.45 * c_df["shap_share_pct"]
        + 0.35 * c_df["perm_share_pct"]
        + 0.20 * c_df["xgb_gain_share_pct"]
    )
    c_df["mean_rank_3methods"] = (c_df["rank_shap"] + c_df["rank_perm"] + c_df["rank_gain"]) / 3.0
    c_df = c_df.sort_values(["consensus_score_pct", "shap_mean_abs"], ascending=False).reset_index(drop=True)
    c_df["consensus_rank"] = np.arange(1, len(c_df) + 1)

    rho_sp, _ = stats.spearmanr(c_df["shap_mean_abs"], c_df["perm_auc_drop_mean"])
    rho_sg, _ = stats.spearmanr(c_df["shap_mean_abs"], c_df["xgb_gain_mean"])
    rho_pg, _ = stats.spearmanr(c_df["perm_auc_drop_mean"], c_df["xgb_gain_mean"])

    consensus_by_ms[snap_label] = c_df
    global_rank_agreement_4ms[snap_label] = {
        "spearman_shap_vs_perm": round(float(rho_sp), 4),
        "spearman_shap_vs_gain": round(float(rho_sg), 4),
        "spearman_perm_vs_gain": round(float(rho_pg), 4),
    }
    print(f"[{snap_label}] N={len(df_m):,} | Top-1 Consensus: {c_df.iloc[0]['feature_name']} ({c_df.iloc[0]['consensus_score_pct']:.2f}%) | Spearman(S,P)={rho_sp:.4f}, (S,G)={rho_sg:.4f}, (P,G)={rho_pg:.4f}")

consensus_df = consensus_by_ms["25%"].drop(columns=["milestone"]).copy()
consensus_4ms_df = pd.concat([consensus_by_ms[m] for m in MILESTONES], ignore_index=True)
rho_shap_perm = global_rank_agreement_4ms["25%"]["spearman_shap_vs_perm"]
rho_shap_gain = global_rank_agreement_4ms["25%"]["spearman_shap_vs_gain"]
rho_perm_gain = global_rank_agreement_4ms["25%"]["spearman_perm_vs_gain"]

logger.info(f"Hoàn tất Global XAI 3 phương pháp qua 4 mốc ({time.time() - t_global:.1f}s)")

# Hiển thị bảng tiến hóa Top 5 Consensus qua 4 mốc
for m in MILESTONES:
    print(f"\n=== TOP 5 ĐỒNG THUẬN TOÀN CỤC — MỐC {m} ===")
    display(consensus_by_ms[m].head(5)[[
        "consensus_rank", "feature_name", "sedl_dimension", "shap_mean_abs",
        "perm_auc_drop_mean", "xgb_gain_mean", "consensus_score_pct",
    ]])

# Dọn bỏ 6 tên tệp hình cũ (38..43 bản gộp dày đặc) trước khi xuất bộ 14 hình thoáng rõ (38..51)
OBSOLETE_WEEK8_FIGS = [
    "38_week8_global_feature_importance_comparison.png",
    "39_week8_shap_summary_beeswarm_and_sedl.png",
    "40_week8_pdp_ice_and_shap_dependence.png",
    "41_week8_local_xai_case_studies_waterfall.png",
    "42_week8_lime_vs_shap_local_fidelity.png",
    "43_week8_demographic_fairness_audit.png",
]
for old_fn in OBSOLETE_WEEK8_FIGS:
    old_p = FIGURES_DIR / old_fn
    if old_p.exists():
        old_p.unlink()

# -------------------------------------------------------------------------
# HÌNH 38: 38_week8_global_xai_3methods_4milestones.png (2x2 = 4 biểu đồ)
# -------------------------------------------------------------------------
panel_tags = ["(A)", "(B)", "(C)", "(D)"]
fig, axes = plt.subplots(2, 2, figsize=(20.5, 13.8), gridspec_kw={"hspace": 0.34, "wspace": 0.44})
axes_flat = axes.ravel()

for idx_m, snap_label in enumerate(MILESTONES):
    ax = axes_flat[idx_m]
    c_df = consensus_by_ms[snap_label]
    agr = global_rank_agreement_4ms[snap_label]
    n_m = len(ms_results[snap_label]["df"])

    top10_s = c_df.sort_values("shap_mean_abs", ascending=False).head(10).iloc[::-1].reset_index(drop=True)
    y_10 = np.arange(len(top10_s))
    h = 0.26
    ax.barh(y_10 + h, top10_s["shap_share_pct"], height=h, color="#2980b9", edgecolor="black", linewidth=0.7, label="Exact TreeSHAP Mean |φ| (%)")
    ax.barh(y_10, top10_s["perm_share_pct"], height=h, color="#27ae60", edgecolor="black", linewidth=0.7, label="5-Fold Perm ROC-AUC (%)")
    ax.barh(y_10 - h, top10_s["xgb_gain_share_pct"], height=h, color="#e67e22", edgecolor="black", linewidth=0.7, label="XGBoost Built-in Gain (%)")
    ax.set_yticks(y_10)
    ax.set_yticklabels(top10_s["feature_name"], fontsize=10.2, fontweight="bold")
    ax.tick_params(axis="x", labelsize=9.8)
    ax.set_xlabel("Tỷ trọng Đóng góp Toàn cục Chuẩn hóa (%)", fontsize=10.4, fontweight="bold")
    ax.set_title(
        f"{panel_tags[idx_m]} Mốc {snap_label} (N = {n_m:,})\n"
        f"Spearman ρ(SHAP, Perm) = {agr['spearman_shap_vs_perm']:.3f} | ρ(SHAP, Gain) = {agr['spearman_shap_vs_gain']:.3f} | ρ(Perm, Gain) = {agr['spearman_perm_vs_gain']:.3f}",
        fontsize=11.4, fontweight="bold",
    )
    ax.grid(axis="x", alpha=0.32)

handles_38, labels_38 = axes_flat[0].get_legend_handles_labels()
fig.legend(handles_38, labels_38, loc="upper center", bbox_to_anchor=(0.5, 0.962), ncol=3, fontsize=10.5, frameon=True)
fig.subplots_adjust(top=0.895)
fig.suptitle(
    "Đối sánh 3 Kỹ thuật XAI Toàn cục (Top 10 Exact TreeSHAP) qua 4 Mốc (25%, 50%, 75%, 100% Dynamic Active)",
    fontsize=14.2, fontweight="bold", y=0.992,
)
fig38_path = FIGURES_DIR / "38_week8_global_xai_3methods_4milestones.png"
fig.savefig(fig38_path, dpi=260, bbox_inches="tight", pad_inches=0.20)
plt.close(fig)
print(f"Đã xuất Hình 38: {fig38_path}")

# -------------------------------------------------------------------------
# HÌNH 39: 39_week8_top10_sedl_consensus_4milestones.png (2x2 = 4 biểu đồ)
# -------------------------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(21.0, 13.8), gridspec_kw={"hspace": 0.34, "wspace": 0.50})
axes_flat = axes.ravel()

for idx_m, snap_label in enumerate(MILESTONES):
    ax = axes_flat[idx_m]
    c_df = consensus_by_ms[snap_label]
    n_m = len(ms_results[snap_label]["df"])

    top10_c = c_df.head(10).iloc[::-1].reset_index(drop=True)
    y_c = np.arange(len(top10_c))
    cols_c = [SEDL_COLORS[d] for d in top10_c["sedl_dimension"]]
    ax.barh(y_c, top10_c["consensus_score_pct"], height=0.64, color=cols_c, edgecolor="black", linewidth=0.8)
    ax.set_yticks(y_c)
    ax.set_yticklabels([f"#{r} {fn}" for r, fn in zip(top10_c["consensus_rank"], top10_c["feature_name"])], fontsize=10.0, fontweight="bold")
    max_c = float(top10_c["consensus_score_pct"].max())
    ax.set_xlim(0, max_c * 1.56)
    ax.tick_params(axis="x", labelsize=9.8)
    for yp, row in top10_c.iterrows():
        ax.text(
            row["consensus_score_pct"] + max_c * 0.02, yp,
            f"{row['consensus_score_pct']:.2f}%  [S:#{row['rank_shap']}, P:#{row['rank_perm']}, G:#{row['rank_gain']}]",
            va="center", fontsize=9.1, fontweight="bold", color="#1f2937",
        )
    ax.set_xlabel("Điểm Đồng thuận Tổng hợp (%)", fontsize=10.4, fontweight="bold")
    ax.set_title(f"{panel_tags[idx_m]} Top 10 Đặc trưng Đồng thuận SEDL — Mốc {snap_label} (N = {n_m:,})", fontsize=11.4, fontweight="bold")
    ax.grid(axis="x", alpha=0.32)

sedl_handles = [mpatches.Patch(facecolor=SEDL_COLORS[d], edgecolor="black", label=d) for d in SEDL_DIM_ORDER]
fig.legend(handles=sedl_handles, loc="upper center", bbox_to_anchor=(0.5, 0.962), ncol=4, fontsize=10.5, title="Chiều Khung SEDL", title_fontsize=10.5, frameon=True)
fig.subplots_adjust(top=0.895)
fig.suptitle(
    "Xếp hạng Top 10 Đặc trưng Đồng thuận theo 4 Chiều Khung SEDL qua 4 Mốc (25%, 50%, 75%, 100% Dynamic Active)",
    fontsize=14.2, fontweight="bold", y=0.992,
)
fig39_path = FIGURES_DIR / "39_week8_top10_sedl_consensus_4milestones.png"
fig.savefig(fig39_path, dpi=260, bbox_inches="tight", pad_inches=0.20)
plt.close(fig)
print(f"Đã xuất Hình 39: {fig39_path}")

[25%] N=26,248 | Top-1 Consensus: weighted_avg_score (13.12%) | Spearman(S,P)=0.8296, (S,G)=0.7810, (P,G)=0.7027


[50%] N=24,601 | Top-1 Consensus: submission_rate (18.47%) | Spearman(S,P)=0.7280, (S,G)=0.7134, (P,G)=0.5943


[75%] N=23,159 | Top-1 Consensus: weighted_avg_score (16.20%) | Spearman(S,P)=0.7410, (S,G)=0.7374, (P,G)=0.6581


[100%] N=22,522 | Top-1 Consensus: max_inactivity_streak (12.94%) | Spearman(S,P)=0.7849, (S,G)=0.7991, (P,G)=0.7281
2026-11-25 21:38:08,427 [INFO] Hoàn tất Global XAI 3 phương pháp qua 4 mốc (73.3s)



=== TOP 5 ĐỒNG THUẬN TOÀN CỤC — MỐC 25% ===


,consensus_rank,feature_name,sedl_dimension,shap_mean_abs,perm_auc_drop_mean,xgb_gain_mean,consensus_score_pct
0,1,weighted_avg_score,Cognitive,0.3669,0.0202,243.3666,13.1221
1,2,avg_score,Cognitive,0.3195,0.0185,170.3429,11.3552
2,3,course_progress_submission_rate,Behavioural,0.1747,0.0091,60.7117,5.6032
3,4,submission_rate,Behavioural,0.1795,0.0051,93.7732,4.6287
4,5,code_module_GGG,Demographic & Context,0.1386,0.0076,39.1597,4.5013


... (4 outputs left out)


=== TOP 5 ĐỒNG THUẬN TOÀN CỤC — MỐC 100% ===


,consensus_rank,feature_name,sedl_dimension,shap_mean_abs,perm_auc_drop_mean,xgb_gain_mean,consensus_score_pct
0,1,max_inactivity_streak,Behavioural,0.7427,0.0083,537.2053,12.9422
1,2,submission_rate,Behavioural,0.6289,0.0121,151.8745,11.5990
2,3,days_since_last_activity,Behavioural,0.6689,0.0094,236.5970,11.0146
3,4,weighted_avg_score,Cognitive,0.4186,0.0100,118.6988,8.8211
4,5,course_progress_submission_rate,Behavioural,0.2681,0.0014,607.2931,6.5715


Đã xuất Hình 38: C:\Users\mrore\Project_I\reports\figures\38_week8_global_xai_3methods_4milestones.png


Đã xuất Hình 39: C:\Users\mrore\Project_I\reports\figures\39_week8_top10_sedl_consensus_4milestones.png


## 3. Phân phối Exact TreeSHAP Beeswarm qua Cả 4 Mốc (`25%`, `50%`, `75%`, `100%`) & Tiến hóa 4 Chiều SEDL
Xuất **Hình 40** (`2×2`: 4 biểu đồ Exact TreeSHAP Beeswarm Top 12 đặc trưng cho `25%`, `50%`, `75%`, `100%`) và **Hình 41** (`1×2`: tiến hóa tỷ trọng % và cường độ tuyệt đối $\sum|\phi|$ theo 4 chiều SEDL song hành cùng `ROC-AUC`).

In [4]:
t_ms = time.time()
milestone_sedl_rows = []
milestone_top5_dict = {}

for snap_label in MILESTONES:
    res_m = ms_results[snap_label]
    df_m = res_m["df"]
    shap_m = res_m["oof_shap"]
    auc_m = float(res_m["fold_df"]["roc_auc"].mean())

    mean_abs_m = np.mean(np.abs(shap_m), axis=0)
    dim_abs = {d: 0.0 for d in SEDL_DIM_ORDER}
    for fn, val in zip(FEAT_NAMES_114, mean_abs_m):
        dim_abs[SEDL_4D_MAP[fn]] += float(val)
    total_abs = sum(dim_abs.values())
    row_sedl = {
        "snapshot": snap_label,
        "n_cohort": int(len(df_m)),
        "roc_auc_5fold": round(auc_m, 4),
        "total_mean_abs_shap": round(total_abs, 4),
    }
    for d in SEDL_DIM_ORDER:
        slug = d.lower().replace(" & ", "_").replace(" ", "_")
        row_sedl[f"{slug}_abs"] = round(dim_abs[d], 4)
        row_sedl[f"{slug}_pct"] = round(100.0 * dim_abs[d] / total_abs, 2)
    milestone_sedl_rows.append(row_sedl)

    top5_idx = np.argsort(mean_abs_m)[::-1][:5]
    milestone_top5_dict[snap_label] = [
        {"feature": FEAT_NAMES_114[i], "sedl": SEDL_4D_MAP[FEAT_NAMES_114[i]], "mean_abs_shap": round(float(mean_abs_m[i]), 4)}
        for i in top5_idx
    ]

sedl_evolution_df = pd.DataFrame(milestone_sedl_rows)
logger.info(f"Hoàn tất tổng hợp SEDL 4 chiều qua 4 mốc ({time.time() - t_ms:.1f}s)")
print("=== TIẾN HÓA ĐÓNG GÓP 4 CHIỀU KHUNG SEDL QUA 4 MỐC (DYNAMIC ACTIVE) ===")
display(sedl_evolution_df[["snapshot", "n_cohort", "roc_auc_5fold", "total_mean_abs_shap", "behavioural_pct", "cognitive_pct", "social_pct", "demographic_context_pct"]])

# -------------------------------------------------------------------------
# HÌNH 40: 40_week8_shap_beeswarm_4milestones.png (2x2 = 4 biểu đồ Beeswarm)
# -------------------------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(21.0, 14.5), gridspec_kw={"hspace": 0.30, "wspace": 0.42})
axes_flat = axes.ravel()
panel_tags = ["(A)", "(B)", "(C)", "(D)"]
rng_bee = np.random.default_rng(42)
cmap_bee = plt.cm.coolwarm
feat_to_col_idx = {fn: i for i, fn in enumerate(FEAT_NAMES_114)}

for idx_m, snap_label in enumerate(MILESTONES):
    ax_b = axes_flat[idx_m]
    res_m = ms_results[snap_label]
    df_m = res_m["df"]
    shap_m = res_m["oof_shap"]
    X_m = res_m["X_oof_scaled"]
    c_df_m = consensus_by_ms[snap_label]

    bee_idx = rng_bee.choice(len(df_m), size=2500, replace=False)
    top12_feats = c_df_m.sort_values("shap_mean_abs", ascending=False).head(12)["feature_name"].tolist()[::-1]

    for y_pos, fn in enumerate(top12_feats):
        c_idx = feat_to_col_idx[fn]
        sv_pts = shap_m[bee_idx, c_idx]
        fv_pts = X_m.iloc[bee_idx, c_idx].values
        q05, q95 = np.percentile(fv_pts, 5), np.percentile(fv_pts, 95)
        c_norm = np.clip((fv_pts - q05) / (q95 - q05), 0.0, 1.0) if (q95 - q05 > 1e-8) else np.full_like(fv_pts, 0.5)
        jitter = rng_bee.normal(0.0, 0.14, size=len(sv_pts))
        ax_b.scatter(sv_pts, y_pos + jitter, c=c_norm, cmap=cmap_bee, s=11, alpha=0.62, edgecolors="none", rasterized=True)

    ax_b.axvline(0.0, color="#2c3e50", linestyle="--", linewidth=1.3)
    ax_b.set_yticks(np.arange(len(top12_feats)))
    ax_b.set_yticklabels(top12_feats, fontsize=10.2, fontweight="bold")
    ax_b.tick_params(axis="x", labelsize=9.8)
    ax_b.set_xlabel("Exact TreeSHAP φ (log-odds | <0: An toàn, >0: Nguy cơ)", fontsize=10.2, fontweight="bold")
    ax_b.set_title(f"{panel_tags[idx_m]} Beeswarm Top 12 — Mốc {snap_label} (N = {len(df_m):,})", fontsize=11.6, fontweight="bold")
    ax_b.grid(axis="x", alpha=0.32)

sm_bee = plt.cm.ScalarMappable(cmap=cmap_bee, norm=plt.Normalize(vmin=0, vmax=1))
sm_bee.set_array([])
cax = fig.add_axes([0.25, 0.035, 0.50, 0.020])
cbar = fig.colorbar(sm_bee, cax=cax, orientation="horizontal")
cbar.set_ticks([0.05, 0.95])
cbar.set_ticklabels(["Giá trị Đặc trưng Thấp", "Giá trị Đặc trưng Cao"])
cbar.ax.tick_params(labelsize=10.5)
fig.subplots_adjust(bottom=0.10, top=0.92)
fig.suptitle(
    "Phân phối Exact TreeSHAP Beeswarm (Top 12 Đặc trưng) qua Cả 4 Mốc (25%, 50%, 75%, 100% Dynamic Active)",
    fontsize=14.2, fontweight="bold", y=0.985,
)
fig40_path = FIGURES_DIR / "40_week8_shap_beeswarm_4milestones.png"
fig.savefig(fig40_path, dpi=260, bbox_inches="tight", pad_inches=0.20)
plt.close(fig)
print(f"Đã xuất Hình 40: {fig40_path}")

# -------------------------------------------------------------------------
# HÌNH 41: 41_week8_sedl_4d_evolution_4milestones.png (1x2 = 2 biểu đồ SEDL)
# -------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(19.0, 7.8), gridspec_kw={"wspace": 0.30})
x_ms = np.arange(len(sedl_evolution_df))

# Panel (A): 100% Stacked Bar tỷ trọng % 4 chiều SEDL
ax_a = axes[0]
bottoms = np.zeros(len(sedl_evolution_df))
slug_pct_map = {
    "Behavioural": "behavioural_pct",
    "Cognitive": "cognitive_pct",
    "Social": "social_pct",
    "Demographic & Context": "demographic_context_pct",
}
for dim in SEDL_DIM_ORDER:
    vals = sedl_evolution_df[slug_pct_map[dim]].values
    ax_a.bar(x_ms, vals, bottom=bottoms, width=0.55, color=SEDL_COLORS[dim], edgecolor="black", linewidth=0.9, label=dim)
    for xi, (b, v) in enumerate(zip(bottoms, vals)):
        fs_txt = 10.5 if v >= 5.0 else 8.8
        ax_a.text(xi, b + v / 2.0, f"{v:.1f}%", ha="center", va="center", fontsize=fs_txt, fontweight="bold", color="white")
    bottoms += vals

ax_a.set_xticks(x_ms)
ax_a.set_xticklabels([f"{r['snapshot']}\n(N={r['n_cohort']:,})" for _, r in sedl_evolution_df.iterrows()], fontsize=10.4, fontweight="bold")
ax_a.set_ylim(0, 118)
ax_a.set_ylabel("Tỷ trọng Tổng Đóng góp Exact TreeSHAP (%)", fontsize=10.8, fontweight="bold")
ax_a.set_title("(A) Tỷ trọng Đóng góp (%) theo 4 Chiều SEDL (25% → 100%)", fontsize=12.0, fontweight="bold")
ax_a.legend(loc="upper center", bbox_to_anchor=(0.5, 0.99), ncol=2, fontsize=9.6, frameon=True)
ax_a.grid(axis="y", alpha=0.35)

# Panel (B): Cường độ tuyệt đối E[sum |phi|] từng chiều SEDL + Tổng sum|phi| & ROC-AUC
ax_b = axes[1]
slug_abs_map = {
    "Behavioural": "behavioural_abs",
    "Cognitive": "cognitive_abs",
    "Social": "social_abs",
    "Demographic & Context": "demographic_context_abs",
}
w_b = 0.18
for i_d, dim in enumerate(SEDL_DIM_ORDER):
    vals_abs = sedl_evolution_df[slug_abs_map[dim]].values
    ax_b.bar(x_ms + (i_d - 1.5) * w_b, vals_abs, width=w_b, color=SEDL_COLORS[dim], edgecolor="black", linewidth=0.7, label=f"{dim} |φ|")

ax_b.plot(x_ms, sedl_evolution_df["total_mean_abs_shap"], "o-", color="#1e293b", linewidth=2.4, markersize=7, label="Tổng Σ|φ| toàn mô hình")
for xi, tot_v in enumerate(sedl_evolution_df["total_mean_abs_shap"]):
    ax_b.text(xi, tot_v + 0.16, f"Σ={tot_v:.2f}", ha="center", va="bottom", fontsize=9.6, fontweight="bold", color="#1e293b")

ax_b_auc = ax_b.twinx()
ax_b_auc.plot(x_ms, sedl_evolution_df["roc_auc_5fold"], "s--", color="#c0392b", linewidth=2.2, markersize=7, label="5-Fold ROC-AUC")
for xi, auc_v in enumerate(sedl_evolution_df["roc_auc_5fold"]):
    ax_b_auc.text(xi + 0.14, auc_v - 0.010, f"{auc_v:.4f}", ha="left", va="top", fontsize=9.2, fontweight="bold", color="#c0392b")

ax_b.set_xticks(x_ms)
ax_b.set_xticklabels([f"{r['snapshot']}\n(N={r['n_cohort']:,})" for _, r in sedl_evolution_df.iterrows()], fontsize=10.4, fontweight="bold")
ax_b.set_ylim(0, 7.6)
ax_b_auc.set_ylim(0.80, 1.00)
ax_b.set_ylabel("Cường độ Đóng góp Tuyệt đối E[Σ|φ|] (log-odds)", fontsize=10.6, fontweight="bold")
ax_b_auc.set_ylabel("5-Fold Test ROC-AUC", fontsize=10.6, fontweight="bold", color="#c0392b")
ax_b.set_title("(B) Tiến hóa Cường độ Tuyệt đối Σ|φ| & ROC-AUC qua 4 Mốc", fontsize=12.0, fontweight="bold")

h1, l1 = ax_b.get_legend_handles_labels()
h2, l2 = ax_b_auc.get_legend_handles_labels()
ax_b.legend(h1 + h2, l1 + l2, loc="upper left", ncol=2, fontsize=8.8, frameon=True)
ax_b.grid(axis="y", alpha=0.35)

fig.suptitle(
    "Tiến hóa Đóng góp Tương đối & Tuyệt đối theo Khung Lý thuyết SEDL (25% → 50% → 75% → 100%)",
    fontsize=14.0, fontweight="bold", y=0.99,
)
fig41_path = FIGURES_DIR / "41_week8_sedl_4d_evolution_4milestones.png"
fig.savefig(fig41_path, dpi=260, bbox_inches="tight", pad_inches=0.20)
plt.close(fig)
print(f"Đã xuất Hình 41: {fig41_path}")

2026-11-25 21:38:10,739 [INFO] Hoàn tất tổng hợp SEDL 4 chiều qua 4 mốc (0.0s)


=== TIẾN HÓA ĐÓNG GÓP 4 CHIỀU KHUNG SEDL QUA 4 MỐC (DYNAMIC ACTIVE) ===


,snapshot,n_cohort,roc_auc_5fold,total_mean_abs_shap,behavioural_pct,cognitive_pct,social_pct,demographic_context_pct
0,25%,26248,0.8480,3.7888,40.8200,36.0800,2.0800,21.0200
1,50%,24601,0.9068,4.6299,52.8800,32.2000,2.2200,12.7000
2,75%,23159,0.9457,5.4936,56.0000,31.8900,3.1300,8.9800
3,100%,22522,0.9655,6.4703,59.3800,30.5100,2.2700,7.8400


Đã xuất Hình 40: C:\Users\mrore\Project_I\reports\figures\40_week8_shap_beeswarm_4milestones.png


Đã xuất Hình 41: C:\Users\mrore\Project_I\reports\figures\41_week8_sedl_4d_evolution_4milestones.png


## 4. Tác động Biên Phi tuyến (PDP/ICE & SHAP Dependence) qua Cả 4 Mốc (`25%`, `50%`, `75%`, `100%`)
Định lượng tác động biên trên không gian xác suất ($\text{PDP} + \text{ICE}$, mẫu 800 sinh viên mỗi mốc) và không gian log-odds Exact TreeSHAP cho **5 đặc trưng hành vi – nhận thức chủ chốt** xuyên suốt cả 4 mốc (`25% -> 100%`), tách gọn thành 3 hình thoáng rõ: **Hình 42** (`weighted_avg_score` & `submission_rate`, `2×2`), **Hình 43** (`days_since_last_activity` & `max_inactivity_streak`, `2×2`), và **Hình 44** (`consistency_score`, `1×2`).

In [5]:
TOP5_PDP_FEATURES = [
    ("weighted_avg_score", "Điểm TB có trọng số (0–100)", (0.0, 100.0)),
    ("submission_rate", "Tỷ lệ nộp bài đúng tiến độ (0–1)", (0.0, 1.0)),
    ("days_since_last_activity", "Số ngày vắng mặt gần nhất (ngày)", (0.0, 120.0)),
    ("consistency_score", "Độ đều đặn học tập [0–1]", (0.0, 1.0)),
    ("max_inactivity_streak", "Chuỗi ngày ngưng dài nhất (ngày)", (0.0, 120.0)),
]

def raw_to_scaled_val(raw_grid: np.ndarray, col_name: str, state: Dict[str, Any]) -> np.ndarray:
    c_idx = state["cols_to_scale"].index(col_name)
    center = float(state["scaler"].center_[c_idx])
    scale = float(state["scaler"].scale_[c_idx])
    vals = raw_grid.astype(float).copy()
    if col_name in state["bounds"]:
        low, high = state["bounds"][col_name]
        vals = np.clip(vals, low, high)
    if col_name in state["skewed_cols"] and "click" in col_name and "ratio" not in col_name and "per" not in col_name and "cv" not in col_name:
        vals = np.log1p(np.maximum(0.0, vals))
    return (vals - center) / (scale if abs(scale) > 1e-12 else 1.0)


def find_zero_crossover(xs: List[float], ys: List[float]) -> Any:
    for i in range(len(xs) - 1):
        y0, y1 = ys[i], ys[i + 1]
        if (y0 == 0.0) or (y0 * y1 < 0.0):
            if abs(y1 - y0) < 1e-12:
                return round(float(xs[i]), 2)
            x_cross = xs[i] + (0.0 - y0) * (xs[i + 1] - xs[i]) / (y1 - y0)
            return round(float(x_cross), 2)
    return None


t_pdp = time.time()
rng_pdp = np.random.default_rng(42)
pdp_4ms_summary: Dict[str, Dict[str, Any]] = {m: {} for m in MILESTONES}
pdp_curves_store: Dict[str, Dict[str, Any]] = {m: {} for m in MILESTONES}

for snap_label in MILESTONES:
    res_m = ms_results[snap_label]
    df_m = res_m["df"]
    full_state_m = res_m["full_state"]
    full_model_m = res_m["full_model"]
    oof_shap_m = res_m["oof_shap"]

    pdp_sample_idx = rng_pdp.choice(len(df_m), size=800, replace=False)
    X_pdp_base = engine.transform_with_state(df_m.iloc[pdp_sample_idx], full_state_m)[FEAT_NAMES_114].copy()
    dep_idx = rng_pdp.choice(len(df_m), size=2500, replace=False)

    for feat, _, (xmin, xmax) in TOP5_PDP_FEATURES:
        # Giữ khoảng [0, 75] cho days_since_last_activity tại 25% như quy chuẩn gốc, mở rộng tới 120 ở các mốc sau hoặc dùng chung [0, xmax]
        xmax_use = 75.0 if (snap_label == "25%" and feat == "days_since_last_activity") else xmax
        grid_raw = np.linspace(xmin, xmax_use, 41)
        grid_scaled = raw_to_scaled_val(grid_raw, feat, full_state_m)

        ice_curves = np.zeros((len(X_pdp_base), len(grid_raw)), dtype=float)
        X_tmp = X_pdp_base.copy()
        for gi, gs in enumerate(grid_scaled):
            X_tmp[feat] = gs
            ice_curves[:, gi] = full_model_m.predict_proba(X_tmp)[:, 1]
        pdp_mean = ice_curves.mean(axis=0)
        pdp_q25 = np.percentile(ice_curves, 25, axis=0)
        pdp_q75 = np.percentile(ice_curves, 75, axis=0)

        grads = np.abs(np.gradient(pdp_mean, grid_raw))
        steep_idx = int(np.argmax(grads))
        inflection_raw = float(grid_raw[steep_idx])

        # SHAP Dependence binned curve trên 2,500 OOF
        raw_x = df_m[feat].fillna(0.0).values[dep_idx]
        raw_x_clipped = np.clip(raw_x, xmin, xmax_use)
        c_idx = FEAT_NAMES_114.index(feat)
        shap_y = oof_shap_m[dep_idx, c_idx]
        bins = np.linspace(xmin, xmax_use, 13)
        bin_centers, bin_means = [], []
        for b0, b1 in zip(bins[:-1], bins[1:]):
            m_b = (raw_x_clipped >= b0) & (raw_x_clipped <= b1)
            if m_b.sum() >= 10:
                bin_centers.append(0.5 * (b0 + b1))
                bin_means.append(float(np.mean(shap_y[m_b])))
        shap_cross = find_zero_crossover(bin_centers, bin_means)

        pdp_4ms_summary[snap_label][feat] = {
            "pdp_min": round(float(pdp_mean.min()), 4),
            "pdp_max": round(float(pdp_mean.max()), 4),
            "pdp_range": round(float(pdp_mean.max() - pdp_mean.min()), 4),
            "inflection_raw_x": round(inflection_raw, 2),
            "pdp_at_inflection": round(float(pdp_mean[steep_idx]), 4),
            "shap_zero_crossover_x": shap_cross,
        }
        pdp_curves_store[snap_label][feat] = {
            "grid_raw": grid_raw,
            "pdp_mean": pdp_mean,
            "pdp_q25": pdp_q25,
            "pdp_q75": pdp_q75,
            "raw_x_sample": raw_x_clipped,
            "shap_y_sample": shap_y,
            "bin_centers": bin_centers,
            "bin_means": bin_means,
        }

# Alias 25% tương thích ngược
pdp_summary_info = {
    k: {kk: vv for kk, vv in pdp_4ms_summary["25%"][k].items() if kk != "shap_zero_crossover_x"}
    for k in ["weighted_avg_score", "submission_rate", "days_since_last_activity", "consistency_score"]
}
logger.info(f"Hoàn tất PDP/ICE & SHAP Dependence 5 biến x 4 mốc ({time.time() - t_pdp:.1f}s)")

# -------------------------------------------------------------------------
# HÌNH 42, 43, 44: PDP/ICE & SHAP Dependence qua 4 Mốc (tách 2x2, 2x2, 1x2)
# -------------------------------------------------------------------------
def draw_pdp_and_shap_pair(ax_p, ax_s, feat: str, xlabel_vi: str, tag_p: str, tag_s: str) -> None:
    c_25 = pdp_curves_store["25%"][feat]
    c_100 = pdp_curves_store["100%"][feat]
    ax_p.fill_between(c_25["grid_raw"], c_25["pdp_q25"], c_25["pdp_q75"], color=MILESTONE_COLORS["25%"], alpha=0.10, label="ICE IQR (25%)")
    ax_p.fill_between(c_100["grid_raw"], c_100["pdp_q25"], c_100["pdp_q75"], color=MILESTONE_COLORS["100%"], alpha=0.10, label="ICE IQR (100%)")
    ax_s.scatter(c_100["raw_x_sample"], c_100["shap_y_sample"], color="#95a5a6", s=8, alpha=0.18, edgecolors="none", rasterized=True)

    summary_lines_p, summary_lines_s = [], []
    for snap_label in MILESTONES:
        cur = pdp_curves_store[snap_label][feat]
        meta_p = pdp_4ms_summary[snap_label][feat]
        col_m = MILESTONE_COLORS[snap_label]
        ax_p.plot(cur["grid_raw"], cur["pdp_mean"], color=col_m, linewidth=2.5, label=f"{snap_label} (ΔP={meta_p['pdp_range']:.2f})")
        ax_p.axvline(meta_p["inflection_raw_x"], color=col_m, linestyle="--", linewidth=1.1, alpha=0.65)
        summary_lines_p.append(f"{snap_label}: ΔP={meta_p['pdp_range']:.2f}, uốn={meta_p['inflection_raw_x']:.1f}")
        if cur["bin_centers"]:
            ax_s.plot(cur["bin_centers"], cur["bin_means"], color=col_m, linewidth=2.4, marker="o", markersize=4.5, label=f"{snap_label}")
        x0_val = meta_p["shap_zero_crossover_x"]
        summary_lines_s.append(f"{snap_label}: x₀={x0_val if x0_val is not None else 'N/A'}")

    ax_p.set_ylim(0.0, 0.94)
    ax_p.tick_params(labelsize=9.8)
    ax_p.set_xlabel(xlabel_vi, fontsize=10.2, fontweight="bold")
    ax_p.set_ylabel("Xác suất Dự đoán P(At-Risk)", fontsize=10.2, fontweight="bold")
    ax_p.set_title(f"{tag_p} PDP & ICE IQR: {feat}", fontsize=11.4, fontweight="bold")
    ax_p.legend(loc="best", fontsize=8.4, frameon=True)
    ax_p.text(
        0.03, 0.97, "\n".join(summary_lines_p), transform=ax_p.transAxes,
        va="top", ha="left", fontsize=8.3, fontweight="bold",
        bbox=dict(facecolor="white", alpha=0.86, edgecolor="#cbd5e1", boxstyle="round,pad=0.3"),
    )
    ax_p.grid(alpha=0.30)

    ax_s.axhline(0.0, color="#1e293b", linestyle="--", linewidth=1.4)
    ax_s.tick_params(labelsize=9.8)
    ax_s.set_xlabel(xlabel_vi, fontsize=10.2, fontweight="bold")
    ax_s.set_ylabel("Exact TreeSHAP φ (log-odds)", fontsize=10.2, fontweight="bold")
    ax_s.set_title(f"{tag_s} SHAP Dependence & φ=0: {feat}", fontsize=11.4, fontweight="bold")
    pos_tail = bool(c_100["bin_means"] and c_100["bin_means"][-1] > 0.2)
    ax_s.legend(loc="upper left" if pos_tail else "lower left", fontsize=8.8, frameon=True)
    ax_s.text(
        0.97, 0.04 if pos_tail else 0.97, "Ngưỡng φ=0:\n" + "\n".join(summary_lines_s), transform=ax_s.transAxes,
        va="bottom" if pos_tail else "top", ha="right", fontsize=8.3, fontweight="bold",
        bbox=dict(facecolor="white", alpha=0.88, edgecolor="#cbd5e1", boxstyle="round,pad=0.3"),
    )
    ax_s.grid(alpha=0.30)


# Hình 42: weighted_avg_score & submission_rate (2x2)
fig, axes = plt.subplots(2, 2, figsize=(19.0, 12.2), gridspec_kw={"hspace": 0.34, "wspace": 0.25})
draw_pdp_and_shap_pair(axes[0, 0], axes[1, 0], "weighted_avg_score", "Điểm TB có trọng số (0–100)", "(A)", "(C)")
draw_pdp_and_shap_pair(axes[0, 1], axes[1, 1], "submission_rate", "Tỷ lệ nộp bài đúng tiến độ (0–1)", "(B)", "(D)")
fig.suptitle(
    "Tác động Biên Phi tuyến (PDP/ICE & SHAP Dependence) cho Điểm số & Tỷ lệ Nộp bài qua 4 Mốc (25%–100%)",
    fontsize=14.0, fontweight="bold", y=0.985,
)
fig42_path = FIGURES_DIR / "42_week8_pdp_shap_score_submission_4milestones.png"
fig.savefig(fig42_path, dpi=260, bbox_inches="tight", pad_inches=0.20)
plt.close(fig)
print(f"Đã xuất Hình 42: {fig42_path}")

# Hình 43: days_since_last_activity & max_inactivity_streak (2x2)
fig, axes = plt.subplots(2, 2, figsize=(19.0, 12.2), gridspec_kw={"hspace": 0.34, "wspace": 0.25})
draw_pdp_and_shap_pair(axes[0, 0], axes[1, 0], "days_since_last_activity", "Số ngày vắng mặt gần nhất (ngày)", "(A)", "(C)")
draw_pdp_and_shap_pair(axes[0, 1], axes[1, 1], "max_inactivity_streak", "Chuỗi ngày ngưng dài nhất (ngày)", "(B)", "(D)")
fig.suptitle(
    "Tác động Biên Phi tuyến (PDP/ICE & SHAP Dependence) cho Gián đoạn & Ngưng hoạt động qua 4 Mốc (25%–100%)",
    fontsize=14.0, fontweight="bold", y=0.985,
)
fig43_path = FIGURES_DIR / "43_week8_pdp_shap_inactivity_4milestones.png"
fig.savefig(fig43_path, dpi=260, bbox_inches="tight", pad_inches=0.20)
plt.close(fig)
print(f"Đã xuất Hình 43: {fig43_path}")

# Hình 44: consistency_score (1x2)
fig, axes = plt.subplots(1, 2, figsize=(18.5, 6.8), gridspec_kw={"wspace": 0.25})
draw_pdp_and_shap_pair(axes[0], axes[1], "consistency_score", "Độ đều đặn học tập hàng tuần [0–1]", "(A)", "(B)")
fig.suptitle(
    "Tác động Biên Phi tuyến (PDP/ICE & SHAP Dependence) cho Độ đều đặn (consistency_score) qua 4 Mốc (25%–100%)",
    fontsize=13.8, fontweight="bold", y=0.99,
)
fig44_path = FIGURES_DIR / "44_week8_pdp_shap_consistency_4milestones.png"
fig.savefig(fig44_path, dpi=260, bbox_inches="tight", pad_inches=0.20)
plt.close(fig)
print(f"Đã xuất Hình 44: {fig44_path}")

print("Ngưỡng chuyển pha PDP & SHAP=0 across 4 milestones:")
print(json.dumps(pdp_4ms_summary, indent=2, ensure_ascii=False))

2026-11-25 21:38:20,164 [INFO] Hoàn tất PDP/ICE & SHAP Dependence 5 biến x 4 mốc (6.4s)


Đã xuất Hình 42: C:\Users\mrore\Project_I\reports\figures\42_week8_pdp_shap_score_submission_4milestones.png


Đã xuất Hình 43: C:\Users\mrore\Project_I\reports\figures\43_week8_pdp_shap_inactivity_4milestones.png


Đã xuất Hình 44: C:\Users\mrore\Project_I\reports\figures\44_week8_pdp_shap_consistency_4milestones.png
Ngưỡng chuyển pha PDP & SHAP=0 across 4 milestones:
{
  "25%": {
    "weighted_avg_score": {
      "pdp_min": 0.4166,
      "pdp_max": 0.6748,
      "pdp_range": 0.2583,
      "inflection_raw_x": 42.5,
      "pdp_at_inflection": 0.6304,
      "shap_zero_crossover_x": 63.13
    },
    "submission_rate": {
      "pdp_min": 0.4416,
      "pdp_max": 0.5131,
      "pdp_range": 0.0715,
      "inflection_raw_x": 1.0,
      "pdp_at_inflection": 0.4416,
      "shap_zero_crossover_x": 0.9
    },
    "days_since_last_activity": {
      "pdp_min": 0.4397,
      "pdp_max": 0.5456,
      "pdp_range": 0.106,
      "inflection_raw_x": 20.62,
      "pdp_at_inflection": 0.4981,
      "shap_zero_crossover_x": 7.6
    },
    "consistency_score": {
      "pdp_min": 0.3837,
      "pdp_max": 0.4949,
      "pdp_range": 0.1112,
      "inflection_raw_x": 0.9,
      "pdp_at_inflection": 0.4286,
      "shap_zer

## 5. Giải thích Cục bộ (Local XAI): Case Studies (`TP`, `FP`, `TN`, `FN`) qua 4 Mốc & Quỹ đạo Dọc (`25% -> 100%`)
Trích xuất 4 trường hợp đại diện (`TP`, `FP`, `TN`, `FN`) từ dự đoán Out-of-Fold tại **TẤT CẢ 4 MỐC (`25%`, `50%`, `75%`, `100%`)**, kèm theo **Quỹ đạo Dọc (Longitudinal Trajectory)** của SV `#2137413` (`EEE-2013J`), và xuất **4 hình Waterfall `2×2` riêng biệt cho từng mốc** (**Hình 45–48**) với tên biến đầy đủ, phông chữ lớn dễ đọc.

In [6]:
def select_milestone_cases(snap_label: str) -> Dict[str, Dict[str, Any]]:
    res_m = ms_results[snap_label]
    df_m = res_m["df"]
    y_m = res_m["y"]
    oof_probs_m = res_m["oof_probs"]
    oof_preds_m = res_m["oof_preds"]
    oof_folds_m = res_m["oof_folds"]
    oof_taus_m = res_m["oof_taus"]
    oof_shap_m = res_m["oof_shap"]
    oof_base_m = res_m["oof_base"]
    X_m = res_m["X_oof_scaled"]

    meta_df = pd.DataFrame({
        "row_idx": np.arange(len(df_m)),
        "id_student": df_m["id_student"].values,
        "code_module": df_m["code_module"].values,
        "code_presentation": df_m["code_presentation"].values,
        "final_result": df_m["final_result"].values,
        "at_risk": y_m,
        "oof_prob": oof_probs_m,
        "oof_pred": oof_preds_m,
        "fold": oof_folds_m,
        "tau_k": oof_taus_m,
        "total_clicks": df_m["total_clicks"].fillna(0.0).values,
        "weighted_avg_score": df_m["weighted_avg_score"].fillna(0.0).values,
    })

    tp_pool = meta_df[(meta_df["at_risk"] == 1) & (meta_df["oof_pred"] == 1) & (meta_df["oof_prob"] >= 0.78) & (meta_df["oof_prob"] <= 0.90) & (meta_df["total_clicks"] >= 40)].sort_values("oof_prob", ascending=False)
    if tp_pool.empty:
        tp_pool = meta_df[(meta_df["at_risk"] == 1) & (meta_df["oof_pred"] == 1)].sort_values("oof_prob", ascending=False)

    fp_pool = meta_df[(meta_df["at_risk"] == 0) & (meta_df["oof_pred"] == 1) & (meta_df["oof_prob"] >= 0.56) & (meta_df["oof_prob"] <= 0.72) & (meta_df["total_clicks"] >= 30)].sort_values("oof_prob", ascending=False)
    if fp_pool.empty:
        fp_pool = meta_df[(meta_df["at_risk"] == 0) & (meta_df["oof_pred"] == 1)].sort_values("oof_prob", ascending=False)

    tn_pool = meta_df[(meta_df["at_risk"] == 0) & (meta_df["oof_pred"] == 0) & (meta_df["oof_prob"] >= 0.06) & (meta_df["oof_prob"] <= 0.14) & (meta_df["weighted_avg_score"] >= 75)].sort_values("oof_prob", ascending=True)
    if tn_pool.empty:
        tn_pool = meta_df[(meta_df["at_risk"] == 0) & (meta_df["oof_pred"] == 0)].sort_values("oof_prob", ascending=True)

    fn_pool = meta_df[(meta_df["at_risk"] == 1) & (meta_df["oof_pred"] == 0) & (meta_df["oof_prob"] >= 0.16) & (meta_df["oof_prob"] <= 0.30) & (meta_df["weighted_avg_score"] >= 55) & (meta_df["total_clicks"] >= 100)].sort_values("oof_prob", ascending=True)
    if fn_pool.empty:
        fn_pool = meta_df[(meta_df["at_risk"] == 1) & (meta_df["oof_pred"] == 0)].sort_values("oof_prob", ascending=True)

    picks = [
        ("TP", "True Positive — Cảnh báo Đúng", tp_pool.iloc[0]),
        ("FP", "False Positive — Cảnh báo Nhầm", fp_pool.iloc[0]),
        ("TN", "True Negative — An toàn Đúng", tn_pool.iloc[0]),
        ("FN", "False Negative — Bỏ sót Nguy cơ", fn_pool.iloc[0]),
    ]

    out = {}
    for quad, title_vi, meta_row in picks:
        r_idx = int(meta_row["row_idx"])
        raw_row = df_m.iloc[r_idx]
        sv_row = oof_shap_m[r_idx]
        bv = float(oof_base_m[r_idx])
        fx = float(bv + sv_row.sum())
        prob = float(meta_row["oof_prob"])
        tau_k = float(meta_row["tau_k"])

        top8_idx = np.argsort(np.abs(sv_row))[::-1][:8]
        top8_list = []
        for j in top8_idx:
            fn = FEAT_NAMES_114[j]
            rv = raw_row[fn] if fn in raw_row.index else X_m.iloc[r_idx, j]
            rv_str = "NaN" if pd.isna(rv) else (f"{rv:.2f}" if isinstance(rv, (float, np.floating, int, np.integer)) else str(rv))
            top8_list.append({
                "feature": fn,
                "sedl_dimension": SEDL_4D_MAP[fn],
                "raw_value": rv_str,
                "scaled_value": round(float(X_m.iloc[r_idx, j]), 4),
                "shap_value": round(float(sv_row[j]), 4),
            })
        other_shap = float(sv_row.sum() - sum(item["shap_value"] for item in top8_list))
        out[quad] = {
            "milestone": snap_label,
            "quadrant": quad,
            "title_vi": title_vi,
            "row_idx": r_idx,
            "id_student": int(meta_row["id_student"]),
            "code_module": str(meta_row["code_module"]),
            "code_presentation": str(meta_row["code_presentation"]),
            "final_result": str(meta_row["final_result"]),
            "at_risk": int(meta_row["at_risk"]),
            "fold": int(meta_row["fold"]),
            "fold_tau": tau_k,
            "oof_prob": round(prob, 4),
            "oof_pred": int(meta_row["oof_pred"]),
            "base_value_log_odds": round(bv, 4),
            "output_log_odds": round(fx, 4),
            "other_106_features_shap_sum": round(other_shap, 4),
            "top8_shap_drivers": top8_list,
        }
    return out


case_studies_4ms: Dict[str, Dict[str, Dict[str, Any]]] = {m: select_milestone_cases(m) for m in MILESTONES}
case_studies_payload = {k: v for k, v in case_studies_4ms["25%"].items()}

# Quỹ đạo dọc (Longitudinal Trajectory 25% -> 100%) cho 1 sinh viên Fail xuất hiện ở cả 4 mốc
common_keys = None
key_maps = {}
for m in MILESTONES:
    df_m = ms_results[m]["df"]
    km = {(int(r.id_student), str(r.code_module), str(r.code_presentation)): i for i, r in enumerate(df_m[["id_student", "code_module", "code_presentation"]].itertuples(index=False))}
    key_maps[m] = km
    common_keys = set(km.keys()) if common_keys is None else (common_keys & set(km.keys()))

longitudinal_candidates = []
for key in sorted(common_keys):
    i25 = key_maps["25%"][key]
    i100 = key_maps["100%"][key]
    if int(ms_results["25%"]["y"][i25]) == 1:
        p25 = float(ms_results["25%"]["oof_probs"][i25])
        p100 = float(ms_results["100%"]["oof_probs"][i100])
        if 0.30 <= p25 <= 0.60 and p100 >= 0.85:
            longitudinal_candidates.append((p100 - p25, key))

longitudinal_candidates.sort(reverse=True)
chosen_long_key = longitudinal_candidates[0][1] if longitudinal_candidates else sorted(common_keys)[0]
longitudinal_trajectory = {
    "id_student": chosen_long_key[0],
    "code_module": chosen_long_key[1],
    "code_presentation": chosen_long_key[2],
    "final_result": str(ms_results["25%"]["df"].iloc[key_maps["25%"][chosen_long_key]]["final_result"]),
    "at_risk": int(ms_results["25%"]["y"][key_maps["25%"][chosen_long_key]]),
    "trajectory": [],
}
for m in MILESTONES:
    idx_m = key_maps[m][chosen_long_key]
    res_m = ms_results[m]
    sv_m = res_m["oof_shap"][idx_m]
    bv_m = float(res_m["oof_base"][idx_m])
    raw_r = res_m["df"].iloc[idx_m]
    top5_j = np.argsort(np.abs(sv_m))[::-1][:5]
    longitudinal_trajectory["trajectory"].append({
        "milestone": m,
        "fold": int(res_m["oof_folds"][idx_m]),
        "fold_tau": float(res_m["oof_taus"][idx_m]),
        "oof_prob": round(float(res_m["oof_probs"][idx_m]), 4),
        "oof_pred": int(res_m["oof_preds"][idx_m]),
        "base_value_log_odds": round(bv_m, 4),
        "output_log_odds": round(float(bv_m + sv_m.sum()), 4),
        "top5_shap_drivers": [
            {
                "feature": FEAT_NAMES_114[j],
                "sedl_dimension": SEDL_4D_MAP[FEAT_NAMES_114[j]],
                "raw_value": str(round(float(raw_r[FEAT_NAMES_114[j]]), 2)) if FEAT_NAMES_114[j] in raw_r.index and pd.notna(raw_r[FEAT_NAMES_114[j]]) and isinstance(raw_r[FEAT_NAMES_114[j]], (int, float, np.integer, np.floating)) else str(round(float(res_m["X_oof_scaled"].iloc[idx_m, j]), 2)),
                "shap_value": round(float(sv_m[j]), 4),
            }
            for j in top5_j
        ],
    })

print("Quỹ đạo dọc 25%->100%:", chosen_long_key, [(t["milestone"], t["oof_prob"], t["oof_pred"]) for t in longitudinal_trajectory["trajectory"]])

# -------------------------------------------------------------------------
# HÌNH 45–48: 45..48_week8_local_waterfall_{25,50,75,100}pct.png (2x2 mỗi mốc)
# -------------------------------------------------------------------------
def plot_milestone_waterfall_2x2(snap_label: str, fig_path: Path) -> None:
    fig, axes = plt.subplots(2, 2, figsize=(20.5, 12.6), gridspec_kw={"hspace": 0.36, "wspace": 0.54})
    axes_flat = axes.ravel()
    quad_specs = [
        ("TP", "(A) True Positive — Cảnh báo Đúng"),
        ("FP", "(B) False Positive — Cảnh báo Nhầm (SV Vượt khó)"),
        ("TN", "(C) True Negative — Xác nhận An toàn Đúng"),
        ("FN", "(D) False Negative — Bỏ sót Nguy cơ"),
    ]
    for ax_i, (quad, q_title) in enumerate(quad_specs):
        ax = axes_flat[ax_i]
        c_dict = case_studies_4ms[snap_label][quad]
        top8_list = c_dict["top8_shap_drivers"]
        bv = float(c_dict["base_value_log_odds"])
        fx = float(c_dict["output_log_odds"])
        prob = float(c_dict["oof_prob"])
        tau_k = float(c_dict["fold_tau"])
        other_106 = float(c_dict["other_106_features_shap_sum"])

        labels_wf = [f"{item['feature']} ({item['raw_value']})" for item in top8_list][::-1]
        vals_wf = [item["shap_value"] for item in top8_list][::-1]
        labels_wf.insert(0, "106 đặc trưng còn lại (∑)")
        vals_wf.insert(0, other_106)

        starts = []
        cur = bv
        for v in vals_wf:
            starts.append(cur)
            cur += v
        colors_wf = ["#e74c3c" if v >= 0 else "#2980b9" for v in vals_wf]
        y_wf = np.arange(len(vals_wf))

        ax.barh(y_wf, vals_wf, left=starts, height=0.64, color=colors_wf, edgecolor="black", linewidth=0.8, zorder=3)
        ax.axvline(bv, color="#7f8c8d", linestyle="--", linewidth=1.2, zorder=2)
        ax.axvline(fx, color="#1e293b", linestyle="-", linewidth=1.8, zorder=2)

        span = max(1.5, max(abs(fx - bv), max(abs(v) for v in vals_wf)) * 2.15)
        xlim_low = min(bv, fx, min(starts), min(np.array(starts) + np.array(vals_wf))) - 0.36 * span
        xlim_high = max(bv, fx, max(starts), max(np.array(starts) + np.array(vals_wf))) + 0.36 * span
        ax.set_xlim(xlim_low, xlim_high)

        txt_box = dict(facecolor="white", alpha=0.88, edgecolor="none", pad=0.4)
        for yp, st_v, val_v in zip(y_wf, starts, vals_wf):
            end_v = st_v + val_v
            if val_v >= 0:
                ax.text(end_v + 0.02 * (xlim_high - xlim_low), yp, f"{val_v:+.3f}", va="center", ha="left", fontsize=9.2, fontweight="bold", color="#922b21", bbox=txt_box, zorder=5)
            else:
                ax.text(end_v - 0.02 * (xlim_high - xlim_low), yp, f"{val_v:+.3f}", va="center", ha="right", fontsize=9.2, fontweight="bold", color="#1b4f72", bbox=txt_box, zorder=5)

        ax.set_yticks(y_wf)
        ax.set_yticklabels(labels_wf, fontsize=9.8, fontweight="bold")
        ax.tick_params(axis="x", labelsize=9.6)
        ax.set_xlabel("Đóng góp Log-odds At-Risk", fontsize=10.2, fontweight="bold")
        ax.set_title(
            f"{q_title}\n"
            f"SV #{c_dict['id_student']} ({c_dict['code_module']}-{c_dict['code_presentation']}) | KQ: {c_dict['final_result']} | "
            f"P={prob:.3f} (τ*={tau_k:.2f}) | E={bv:+.3f}→f={fx:+.3f}",
            fontsize=10.6, fontweight="bold",
        )
        ax.grid(axis="x", alpha=0.30)

    wf_handles = [
        mpatches.Patch(facecolor="#e74c3c", edgecolor="black", label="SHAP φ > 0 (Tăng nguy cơ At-Risk)"),
        mpatches.Patch(facecolor="#2980b9", edgecolor="black", label="SHAP φ < 0 (Giảm nguy cơ / Bảo vệ)"),
        plt.Line2D([0], [0], color="#7f8c8d", linestyle="--", linewidth=1.3, label="Nền kỳ vọng E[f(x)]"),
        plt.Line2D([0], [0], color="#1e293b", linestyle="-", linewidth=1.8, label="Đầu ra f(x) = log-odds"),
    ]
    fig.legend(handles=wf_handles, loc="upper center", bbox_to_anchor=(0.5, 0.962), ncol=4, fontsize=10.2, frameon=True)
    fig.subplots_adjust(top=0.89)
    fig.suptitle(
        f"Giải thích Cục bộ Exact TreeSHAP Waterfall cho 4 Case Studies (TP, FP, TN, FN) — Mốc {snap_label} Dynamic Active OOF",
        fontsize=14.0, fontweight="bold", y=0.992,
    )
    fig.savefig(fig_path, dpi=260, bbox_inches="tight", pad_inches=0.20)
    plt.close(fig)
    print(f"Đã xuất: {fig_path}")


waterfall_fig_map = {
    "25%": FIGURES_DIR / "45_week8_local_waterfall_25pct.png",
    "50%": FIGURES_DIR / "46_week8_local_waterfall_50pct.png",
    "75%": FIGURES_DIR / "47_week8_local_waterfall_75pct.png",
    "100%": FIGURES_DIR / "48_week8_local_waterfall_100pct.png",
}
for snap_label, f_path in waterfall_fig_map.items():
    plot_milestone_waterfall_2x2(snap_label, f_path)

Quỹ đạo dọc 25%->100%: (2137413, 'EEE', '2013J') [('25%', 0.3006, 0), ('50%', 0.8499, 1), ('75%', 0.9944, 1), ('100%', 0.9991, 1)]


Đã xuất: C:\Users\mrore\Project_I\reports\figures\45_week8_local_waterfall_25pct.png


Đã xuất: C:\Users\mrore\Project_I\reports\figures\46_week8_local_waterfall_50pct.png


Đã xuất: C:\Users\mrore\Project_I\reports\figures\47_week8_local_waterfall_75pct.png


Đã xuất: C:\Users\mrore\Project_I\reports\figures\48_week8_local_waterfall_100pct.png


## 6. Đối sánh Local Surrogate LIME với Exact TreeSHAP qua 4 Mốc (`25%`, `50%`, `75%`, `100%`)
Chạy `LocalTabularSurrogateLIME` (`1,200` nhiễu loạn Gaussian trên không gian Train Fold, `Ridge(alpha=1.0)` trên đầu ra log-odds) trên **từng Case Study** và **200 sinh viên OOF phân tầng (`40/Fold × 5 Fold`) tại MỖI MỐC** (`25%, 50%, 75%, 100%`), xuất **Hình 49** (`2×2` đối sánh 4 ca TP trải đều 4 mốc) và **Hình 50** (`2×2` tổng hợp độ trung thực & đồng thuận trên 800 mẫu OOF).

In [7]:
class LocalTabularSurrogateLIME:
    '''Local Surrogate Tabular LIME (Ribeiro et al., 2016) trên không gian đặc trưng chuẩn hóa của Train Fold.'''

    def __init__(
        self,
        X_train: pd.DataFrame,
        feature_names: List[str],
        kernel_width: float = None,
        perturbation_std: float = 0.45,
        random_state: int = 42,
    ):
        self.feature_names = list(feature_names)
        arr = X_train[self.feature_names].values.astype(float)
        self.means = arr.mean(axis=0)
        self.stds = np.where(arr.std(axis=0) > 1e-8, arr.std(axis=0), 1.0)
        p = len(self.feature_names)
        self.kernel_width = float(kernel_width) if kernel_width is not None else 0.75 * np.sqrt(p)
        self.perturbation_std = float(perturbation_std)
        self.rng = np.random.default_rng(random_state)

    def explain_instance(
        self, x_row: np.ndarray, model: xgb.XGBClassifier, n_samples: int = 1200
    ) -> Tuple[np.ndarray, float, float]:
        p = len(self.feature_names)
        z_x = (x_row.astype(float) - self.means) / self.stds
        noise = self.rng.normal(0.0, self.perturbation_std, size=(n_samples - 1, p))
        z_pert = np.vstack([z_x[None, :], z_x[None, :] + noise])
        x_pert = z_pert * self.stds + self.means
        x_pert_df = pd.DataFrame(x_pert, columns=self.feature_names)

        probs = np.clip(model.predict_proba(x_pert_df)[:, 1], 1e-6, 1.0 - 1e-6)
        log_odds = np.log(probs / (1.0 - probs))

        dists = np.linalg.norm(z_pert - z_x[None, :], axis=1)
        weights = np.exp(-(dists ** 2) / (self.kernel_width ** 2))

        reg = Ridge(alpha=1.0, random_state=42)
        reg.fit(z_pert, log_odds, sample_weight=weights)
        pred_local = reg.predict(z_pert)
        r2_w = float(r2_score(log_odds, pred_local, sample_weight=weights))
        contribs = reg.coef_ * z_x
        return contribs, r2_w, float(reg.intercept_)


def compare_lime_shap(lime_vec: np.ndarray, shap_vec: np.ndarray) -> Dict[str, float]:
    top10_l = set(np.argsort(np.abs(lime_vec))[-10:])
    top10_s = set(np.argsort(np.abs(shap_vec))[-10:])
    top20_union = sorted(set(np.argsort(np.abs(shap_vec))[-15:]) | set(np.argsort(np.abs(lime_vec))[-15:]))
    rho_all, _ = stats.spearmanr(lime_vec, shap_vec)
    rho_active, _ = stats.spearmanr(lime_vec[top20_union], shap_vec[top20_union])
    denom = float(np.linalg.norm(lime_vec) * np.linalg.norm(shap_vec))
    cos_sim = float(np.dot(lime_vec, shap_vec) / denom) if denom > 1e-12 else 0.0
    return {
        "spearman_all_114": float(0.0 if np.isnan(rho_all) else rho_all),
        "spearman_active_union": float(0.0 if np.isnan(rho_active) else rho_active),
        "cosine_similarity": cos_sim,
        "top10_overlap": len(top10_l & top10_s) / 10.0,
    }


t_lime = time.time()
lime_eval_by_ms: Dict[str, pd.DataFrame] = {}
lime_fidelity_4ms: Dict[str, Dict[str, Any]] = {}
lime_case_vectors_4ms: Dict[str, Dict[str, np.ndarray]] = {m: {} for m in MILESTONES}

for snap_label in MILESTONES:
    res_m = ms_results[snap_label]
    df_m = res_m["df"]
    y_m = res_m["y"]
    splits_m = res_m["splits"]
    X_oof_m = res_m["X_oof_scaled"]
    oof_shap_m = res_m["oof_shap"]
    fold_models_m = res_m["fold_models"]
    fold_preps_m = res_m["fold_preprocessors"]

    fold_explainers = {}
    for fold, tr_idx, _ in splits_m:
        X_tr_k = engine.transform_with_state(df_m.iloc[tr_idx], fold_preps_m[fold - 1])[FEAT_NAMES_114]
        fold_explainers[fold] = LocalTabularSurrogateLIME(X_tr_k, FEAT_NAMES_114, random_state=42 + fold)

    # 1) Giải thích LIME cho 4 Case Studies của mốc
    for quad, c_dict in case_studies_4ms[snap_label].items():
        r_idx = c_dict["row_idx"]
        f_num = c_dict["fold"]
        c_lime, r2_w, _ = fold_explainers[f_num].explain_instance(X_oof_m.iloc[r_idx].values, fold_models_m[f_num - 1], n_samples=1200)
        comp = compare_lime_shap(c_lime, oof_shap_m[r_idx])
        lime_case_vectors_4ms[snap_label][quad] = c_lime
        top8_l_idx = np.argsort(np.abs(c_lime))[::-1][:8]
        c_dict["lime_local_r2"] = round(r2_w, 4)
        c_dict["lime_vs_shap_spearman_active"] = round(comp["spearman_active_union"], 4)
        c_dict["lime_vs_shap_spearman_all114"] = round(comp["spearman_all_114"], 4)
        c_dict["lime_vs_shap_cosine"] = round(comp["cosine_similarity"], 4)
        c_dict["lime_vs_shap_top10_overlap"] = round(comp["top10_overlap"], 4)
        c_dict["top8_lime_drivers"] = [
            {"feature": FEAT_NAMES_114[j], "lime_weight": round(float(c_lime[j]), 4)}
            for j in top8_l_idx
        ]

    # 2) Đánh giá 200 sinh viên OOF phân tầng (40/fold x 5 fold), n_samples=1200
    eval_rows = []
    rng_l = np.random.default_rng(42)
    for fold, _, te_idx in splits_m:
        model_k = fold_models_m[fold - 1]
        expl_k = fold_explainers[fold]
        y_te = y_m[te_idx]
        pick_pos = rng_l.choice(te_idx[y_te == 1], size=20, replace=False)
        pick_neg = rng_l.choice(te_idx[y_te == 0], size=20, replace=False)
        for r_idx in np.concatenate([pick_pos, pick_neg]):
            c_lime, r2_w, _ = expl_k.explain_instance(X_oof_m.iloc[r_idx].values, model_k, n_samples=1200)
            comp = compare_lime_shap(c_lime, oof_shap_m[r_idx])
            eval_rows.append({
                "milestone": snap_label,
                "row_idx": int(r_idx),
                "fold": int(fold),
                "at_risk": int(y_m[r_idx]),
                "local_r2": r2_w,
                **comp,
            })

    df_eval_m = pd.DataFrame(eval_rows)
    lime_eval_by_ms[snap_label] = df_eval_m
    lime_fidelity_4ms[snap_label] = {
        "n_sampled_students": int(len(df_eval_m)),
        "n_perturbations": 1200,
        "local_r2_mean": round(float(df_eval_m["local_r2"].mean()), 4),
        "local_r2_median": round(float(df_eval_m["local_r2"].median()), 4),
        "local_r2_std": round(float(df_eval_m["local_r2"].std(ddof=1)), 4),
        "spearman_active_mean": round(float(df_eval_m["spearman_active_union"].mean()), 4),
        "spearman_active_median": round(float(df_eval_m["spearman_active_union"].median()), 4),
        "spearman_all114_mean": round(float(df_eval_m["spearman_all_114"].mean()), 4),
        "cosine_similarity_mean": round(float(df_eval_m["cosine_similarity"].mean()), 4),
        "top10_overlap_mean": round(float(df_eval_m["top10_overlap"].mean()), 4),
    }
    print(f"[{snap_label}] LIME 200 OOF -> R2={lime_fidelity_4ms[snap_label]['local_r2_mean']:.4f}, Cos={lime_fidelity_4ms[snap_label]['cosine_similarity_mean']:.4f}, Spearman(Act)={lime_fidelity_4ms[snap_label]['spearman_active_mean']:.4f}, Top10={lime_fidelity_4ms[snap_label]['top10_overlap_mean']:.4f}")

# Alias 25% tương thích ngược
case_studies_payload = {k: v for k, v in case_studies_4ms["25%"].items()}
lime_200_df = lime_eval_by_ms["25%"]
lime_all_df = pd.concat([lime_eval_by_ms[m] for m in MILESTONES], ignore_index=True)

logger.info(f"Hoàn tất LIME qua 4 mốc ({time.time() - t_lime:.1f}s)")

# -------------------------------------------------------------------------
# HÌNH 49: 49_week8_lime_vs_shap_cases_4milestones.png (2x2 = 4 ca TP)
# -------------------------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(20.5, 12.6), gridspec_kw={"hspace": 0.36, "wspace": 0.44})
axes_flat = axes.ravel()
panel_tags = ["(A)", "(B)", "(C)", "(D)"]

for idx_m, ms_lbl in enumerate(MILESTONES):
    ax = axes_flat[idx_m]
    c_dict = case_studies_4ms[ms_lbl]["TP"]
    r_idx = c_dict["row_idx"]
    sv_row = ms_results[ms_lbl]["oof_shap"][r_idx]
    lv_row = lime_case_vectors_4ms[ms_lbl]["TP"]
    top8_f = [item["feature"] for item in c_dict["top8_shap_drivers"]][::-1]
    idx_f = [FEAT_NAMES_114.index(fn) for fn in top8_f]
    y_p = np.arange(len(top8_f))
    h_b = 0.36

    ax.barh(y_p + h_b / 2, sv_row[idx_f], height=h_b, color="#2980b9", edgecolor="black", linewidth=0.7, label="Exact TreeSHAP φ")
    ax.barh(y_p - h_b / 2, lv_row[idx_f], height=h_b, color="#e67e22", edgecolor="black", linewidth=0.7, label="Local Surrogate LIME")
    ax.axvline(0.0, color="#2c3e50", linestyle="--", linewidth=1.2)
    xmin_a = min(float(sv_row[idx_f].min()), float(lv_row[idx_f].min()), 0.0)
    xmax_a = max(float(sv_row[idx_f].max()), float(lv_row[idx_f].max()), 0.0)
    pad_a = max(0.08, 0.14 * (xmax_a - xmin_a))
    ax.set_xlim(xmin_a - pad_a, xmax_a + pad_a)
    ax.set_yticks(y_p)
    ax.set_yticklabels(top8_f, fontsize=10.0, fontweight="bold")
    ax.tick_params(axis="x", labelsize=9.8)
    ax.set_xlabel("Đóng góp Cục bộ (log-odds At-Risk)", fontsize=10.2, fontweight="bold")
    ax.set_title(
        f"{panel_tags[idx_m]} Mốc {ms_lbl} — TP #{c_dict['id_student']} ({c_dict['code_module']})\n"
        f"R² = {c_dict['lime_local_r2']:.3f} | Cosine = {c_dict['lime_vs_shap_cosine']:.3f} | Spearman ρ(Active) = {c_dict['lime_vs_shap_spearman_active']:.3f}",
        fontsize=10.8, fontweight="bold",
    )
    ax.grid(axis="x", alpha=0.30)

h49, l49 = axes_flat[0].get_legend_handles_labels()
fig.legend(h49, l49, loc="upper center", bbox_to_anchor=(0.5, 0.962), ncol=2, fontsize=10.5, frameon=True)
fig.subplots_adjust(top=0.895)
fig.suptitle(
    "Đối sánh Đóng góp Cục bộ LIME vs Exact TreeSHAP trên 4 Trường hợp TP Đại diện (25% → 100%)",
    fontsize=14.0, fontweight="bold", y=0.992,
)
fig49_path = FIGURES_DIR / "49_week8_lime_vs_shap_cases_4milestones.png"
fig.savefig(fig49_path, dpi=260, bbox_inches="tight", pad_inches=0.20)
plt.close(fig)
print(f"Đã xuất Hình 49: {fig49_path}")

# -------------------------------------------------------------------------
# HÌNH 50: 50_week8_lime_vs_shap_fidelity_4milestones.png (2x2 = 800 OOF)
# -------------------------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(19.5, 12.6), gridspec_kw={"hspace": 0.34, "wspace": 0.28})

# (A) Boxplot Local Weighted R^2
ax_r2 = axes[0, 0]
sns.boxplot(data=lime_all_df, x="milestone", y="local_r2", order=MILESTONES, palette=MILESTONE_COLORS, width=0.52, ax=ax_r2)
for xi, ms_lbl in enumerate(MILESTONES):
    mv = lime_fidelity_4ms[ms_lbl]["local_r2_mean"]
    sdv = lime_fidelity_4ms[ms_lbl]["local_r2_std"]
    ax_r2.text(xi, float(lime_eval_by_ms[ms_lbl]["local_r2"].max()) + 0.014, f"{mv:.3f}±{sdv:.3f}", ha="center", va="bottom", fontsize=9.6, fontweight="bold")
ax_r2.set_ylim(0.38, 0.90)
ax_r2.tick_params(labelsize=9.8)
ax_r2.set_xlabel("Mốc Tiến trình (N = 200 OOF/mốc)", fontsize=10.2, fontweight="bold")
ax_r2.set_ylabel("Độ trung thực Cục bộ LIME Weighted R²", fontsize=10.2, fontweight="bold")
ax_r2.set_title("(A) Phân phối LIME Weighted R² qua 4 Mốc", fontsize=11.4, fontweight="bold")
ax_r2.grid(axis="y", alpha=0.32)

# (B) Grouped Boxplot Cosine & Spearman Active
ax_cs = axes[0, 1]
cs_melt = lime_all_df.melt(
    id_vars=["milestone"], value_vars=["cosine_similarity", "spearman_active_union"],
    var_name="Chỉ số", value_name="Giá trị",
)
cs_melt["Chỉ số"] = cs_melt["Chỉ số"].map({"cosine_similarity": "Cosine Similarity", "spearman_active_union": "Spearman ρ (Active)"})
sns.boxplot(data=cs_melt, x="milestone", y="Giá trị", hue="Chỉ số", order=MILESTONES, palette=["#2980b9", "#8e44ad"], width=0.60, ax=ax_cs)
ax_cs.set_ylim(-0.15, 1.05)
ax_cs.tick_params(labelsize=9.8)
ax_cs.set_xlabel("Mốc Tiến trình (N = 200 OOF/mốc)", fontsize=10.2, fontweight="bold")
ax_cs.set_ylabel("Hệ số Tương đồng với Exact TreeSHAP", fontsize=10.2, fontweight="bold")
ax_cs.set_title("(B) Cosine Similarity & Spearman ρ (Active) qua 4 Mốc", fontsize=11.4, fontweight="bold")
ax_cs.legend(loc="lower right", fontsize=9.4, frameon=True)
ax_cs.grid(axis="y", alpha=0.32)

# (C) Grouped Boxplot Top-10 Overlap & Spearman All 114
ax_ov = axes[1, 0]
ov_melt = lime_all_df.melt(
    id_vars=["milestone"], value_vars=["top10_overlap", "spearman_all_114"],
    var_name="Chỉ số", value_name="Giá trị",
)
ov_melt["Chỉ số"] = ov_melt["Chỉ số"].map({"top10_overlap": "Top-10 Overlap Ratio", "spearman_all_114": "Spearman ρ (All 114)"})
sns.boxplot(data=ov_melt, x="milestone", y="Giá trị", hue="Chỉ số", order=MILESTONES, palette=["#16a085", "#e67e22"], width=0.60, ax=ax_ov)
ax_ov.set_ylim(-0.05, 1.02)
ax_ov.tick_params(labelsize=9.8)
ax_ov.set_xlabel("Mốc Tiến trình (N = 200 OOF/mốc)", fontsize=10.2, fontweight="bold")
ax_ov.set_ylabel("Mức độ Đồng thuận Xếp hạng", fontsize=10.2, fontweight="bold")
ax_ov.set_title("(C) Top-10 Overlap Ratio & Spearman ρ (All 114) qua 4 Mốc", fontsize=11.4, fontweight="bold")
ax_ov.legend(loc="upper left", fontsize=9.4, frameon=True)
ax_ov.grid(axis="y", alpha=0.32)

# (D) Heatmap tổng hợp 5 chỉ số
ax_hm = axes[1, 1]
m_specs = [
    ("local_r2_mean", "Local Weighted R²"),
    ("cosine_similarity_mean", "Cosine Similarity"),
    ("spearman_active_mean", "Spearman ρ (Active)"),
    ("top10_overlap_mean", "Top-10 Overlap Ratio"),
    ("spearman_all114_mean", "Spearman ρ (All 114)"),
]
heat_mat = np.array([[lime_fidelity_4ms[ms][k] for ms in MILESTONES] for k, _ in m_specs])
sns.heatmap(
    heat_mat, annot=True, fmt=".3f", cmap="YlGnBu",
    xticklabels=MILESTONES, yticklabels=[lbl for _, lbl in m_specs],
    linewidths=1.0, linecolor="white", annot_kws={"size": 11.2, "weight": "bold"},
    cbar_kws={"label": "Giá trị TB (N = 200 OOF/mốc)"}, ax=ax_hm, vmin=0.25, vmax=0.75,
)
ax_hm.tick_params(labelsize=10.0)
ax_hm.set_xlabel("Mốc Tiến trình (N = 200 OOF/mốc)", fontsize=10.2, fontweight="bold")
ax_hm.set_ylabel("")
ax_hm.set_title("(D) Bảng Nhiệt Tổng hợp 5 Chỉ số LIME vs TreeSHAP (25% → 100%)", fontsize=11.4, fontweight="bold")

fig.suptitle(
    "Kiểm chứng Độ trung thực & Đồng thuận LIME (1,200 nhiễu loạn) vs Exact TreeSHAP trên 800 Sinh viên OOF (200/mốc × 4 mốc)",
    fontsize=14.0, fontweight="bold", y=0.985,
)
fig50_path = FIGURES_DIR / "50_week8_lime_vs_shap_fidelity_4milestones.png"
fig.savefig(fig50_path, dpi=260, bbox_inches="tight", pad_inches=0.20)
plt.close(fig)
print(f"Đã xuất Hình 50: {fig50_path}")

[25%] LIME 200 OOF -> R2=0.6755, Cos=0.5869, Spearman(Act)=0.5351, Top10=0.5040


[50%] LIME 200 OOF -> R2=0.6764, Cos=0.6101, Spearman(Act)=0.5952, Top10=0.5290


[75%] LIME 200 OOF -> R2=0.6744, Cos=0.6026, Spearman(Act)=0.6069, Top10=0.5320


[100%] LIME 200 OOF -> R2=0.6939, Cos=0.5738, Spearman(Act)=0.5896, Top10=0.5645
2026-11-25 21:38:51,097 [INFO] Hoàn tất LIME qua 4 mốc (24.1s)


Đã xuất Hình 49: C:\Users\mrore\Project_I\reports\figures\49_week8_lime_vs_shap_cases_4milestones.png


Đã xuất Hình 50: C:\Users\mrore\Project_I\reports\figures\50_week8_lime_vs_shap_fidelity_4milestones.png


## 7. Kiểm định Công bằng Nhân khẩu học (Demographic Fairness Audit) qua 4 Mốc (`25%`, `50%`, `75%`, `100%`)
Đánh giá toàn diện trên dự đoán 5-Fold OOF (ngưỡng tối ưu $\tau^*_k$) cho **5 thuộc tính nhân khẩu học** (`gender`, `disability`, `age_band`, `imd_band`, `region`) xuyên suốt cả 4 mốc, xuất **Hình 51** (`2×2`).

In [8]:
def compute_subgroup_fairness(
    df_eval: pd.DataFrame, y_true: np.ndarray, y_prob: np.ndarray, y_pred: np.ndarray, attr: str, milestone: str
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    s_attr = df_eval[attr].fillna("Missing").astype(str)
    counts = s_attr.value_counts()
    ref_group = str(counts.index[0])

    rows = []
    for grp, cnt in counts.items():
        mask = (s_attr == grp).values
        yt = y_true[mask]
        yp = y_prob[mask]
        yd = y_pred[mask]
        tn, fp, fn, tp = confusion_matrix(yt, yd, labels=[0, 1]).ravel()
        tpr = float(tp / max(1, tp + fn))
        fpr = float(fp / max(1, fp + tn))
        ppr = float(yd.mean())
        prec = float(tp / max(1, tp + fp))
        f1_v = float(f1_score(yt, yd, zero_division=0))
        auc_v = float(roc_auc_score(yt, yp)) if len(np.unique(yt)) > 1 else np.nan
        brier_v = float(brier_score_loss(yt, yp))
        rows.append({
            "milestone": milestone,
            "attribute": attr,
            "subgroup": str(grp),
            "is_reference": str(grp) == ref_group,
            "n": int(cnt),
            "share_pct": round(100.0 * cnt / len(df_eval), 2),
            "base_rate": round(float(yt.mean()), 4),
            "predicted_positive_rate": round(ppr, 4),
            "tpr_recall": round(tpr, 4),
            "fpr": round(fpr, 4),
            "precision": round(prec, 4),
            "f1": round(f1_v, 4),
            "roc_auc": round(auc_v, 4) if not np.isnan(auc_v) else np.nan,
            "brier_score": round(brier_v, 4),
        })
    sub_df = pd.DataFrame(rows)
    ref_ppr = float(sub_df.loc[sub_df["is_reference"], "predicted_positive_rate"].iloc[0])
    sub_df["disparate_impact"] = (sub_df["predicted_positive_rate"] / max(1e-8, ref_ppr)).round(4)

    summary = {
        "milestone": milestone,
        "attribute": attr,
        "reference_group": ref_group,
        "n_subgroups": int(len(sub_df)),
        "tpr_min": float(sub_df["tpr_recall"].min()),
        "tpr_max": float(sub_df["tpr_recall"].max()),
        "equal_opportunity_gap_delta_tpr": round(float(sub_df["tpr_recall"].max() - sub_df["tpr_recall"].min()), 4),
        "fpr_gap_delta_fpr": round(float(sub_df["fpr"].max() - sub_df["fpr"].min()), 4),
        "roc_auc_gap": round(float(sub_df["roc_auc"].max() - sub_df["roc_auc"].min()), 4),
        "disparate_impact_min": float(sub_df["disparate_impact"].min()),
        "disparate_impact_max": float(sub_df["disparate_impact"].max()),
        "passes_4_5ths_rule": bool((sub_df["disparate_impact"].min() >= 0.80) and (sub_df["disparate_impact"].max() <= 1.25)),
    }
    return sub_df, summary


ATTRS = ["gender", "disability", "age_band", "imd_band", "region"]
fairness_dfs_all = []
fairness_summaries_4ms: Dict[str, Dict[str, Any]] = {}

for snap_label in MILESTONES:
    res_m = ms_results[snap_label]
    fairness_summaries_4ms[snap_label] = {}
    for attr in ATTRS:
        s_df, s_sum = compute_subgroup_fairness(
            res_m["df"], res_m["y"], res_m["oof_probs"], res_m["oof_preds"], attr, snap_label
        )
        fairness_dfs_all.append(s_df)
        fairness_summaries_4ms[snap_label][attr] = s_sum

fairness_audit_df = pd.concat(fairness_dfs_all, ignore_index=True)
fairness_summaries = {
    a: {k: v for k, v in fairness_summaries_4ms["25%"][a].items() if k != "milestone"}
    for a in ATTRS
}

print("=== TIẾN HÓA CHÊNH LỆCH CÔNG BẰNG (ΔTPR, ΔFPR, ΔAUC, 4/5 RULE) QUA 4 MỐC ===")
fair_evo_rows = []
for snap_label in MILESTONES:
    for a in ATTRS:
        fair_evo_rows.append(fairness_summaries_4ms[snap_label][a])
fair_evo_df = pd.DataFrame(fair_evo_rows)
display(fair_evo_df.pivot(index="attribute", columns="milestone", values=["equal_opportunity_gap_delta_tpr", "fpr_gap_delta_fpr", "roc_auc_gap", "passes_4_5ths_rule"])[[
    (c, m) for c in ["equal_opportunity_gap_delta_tpr", "fpr_gap_delta_fpr", "roc_auc_gap", "passes_4_5ths_rule"] for m in MILESTONES
]])

# -------------------------------------------------------------------------
# HÌNH 51: 51_week8_demographic_fairness_audit.png (2x2 = 4 biểu đồ)
# -------------------------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(21.5, 12.6), gridspec_kw={"hspace": 0.34, "wspace": 0.28})

# Panel (A): Evolution of ΔTPR & ΔFPR across 4 milestones for 5 attributes
ax_a = axes[0, 0]
x_a = np.arange(len(ATTRS))
w_a = 0.19
for i_m, ms_lbl in enumerate(MILESTONES):
    sub_e = fair_evo_df[fair_evo_df["milestone"] == ms_lbl].set_index("attribute").loc[ATTRS]
    offset = (i_m - 1.5) * w_a
    ax_a.bar(x_a + offset, sub_e["equal_opportunity_gap_delta_tpr"], width=w_a, color=MILESTONE_COLORS[ms_lbl], edgecolor="black", linewidth=0.6, label=f"{ms_lbl} ΔTPR")
    ax_a.plot(x_a + offset, sub_e["fpr_gap_delta_fpr"], "o", color="black", markersize=5.0)
ax_a.plot([], [], "o", color="black", label="Chấm đen: ΔFPR")
ax_a.set_xticks(x_a)
ax_a.set_xticklabels(ATTRS, fontsize=10.4, fontweight="bold")
ax_a.tick_params(axis="y", labelsize=9.8)
ax_a.set_ylim(0.0, 0.32)
ax_a.set_ylabel("Khoảng cách Sai số (ΔTPR & ΔFPR)", fontsize=10.4, fontweight="bold")
ax_a.set_title("(A) Tiến hóa ΔTPR (Equal Opportunity) & ΔFPR qua 4 Mốc (25% → 100%)", fontsize=11.4, fontweight="bold")
ax_a.legend(loc="upper left", ncol=3, fontsize=8.8, frameon=True)
ax_a.grid(axis="y", alpha=0.32)

# Panel (B): IMD Deprivation Band Disparate Impact across 4 milestones
ax_b = axes[0, 1]
imd_order = ["0-10%", "10-20", "20-30%", "30-40%", "40-50%", "50-60%", "60-70%", "70-80%", "80-90%", "90-100%", "Missing"]
x_imd = np.arange(len(imd_order))
ax_b.axhspan(0.80, 1.25, color="#2ecc71", alpha=0.14, label="Vùng Công bằng 4/5 [0.80, 1.25]")
ax_b.axhline(1.0, color="#2c3e50", linestyle="--", linewidth=1.2)
for ms_lbl in MILESTONES:
    sub_imd = fairness_audit_df[(fairness_audit_df["milestone"] == ms_lbl) & (fairness_audit_df["attribute"] == "imd_band")].copy()
    sub_imd["ord"] = sub_imd["subgroup"].apply(lambda x: imd_order.index(x) if x in imd_order else 99)
    sub_imd = sub_imd.sort_values("ord")
    dtpr_imd = fairness_summaries_4ms[ms_lbl]["imd_band"]["equal_opportunity_gap_delta_tpr"]
    ax_b.plot(x_imd, sub_imd["disparate_impact"], "o-", color=MILESTONE_COLORS[ms_lbl], linewidth=2.2, markersize=5.5, label=f"{ms_lbl} (ΔTPR={dtpr_imd:.3f})")
ax_b.set_xticks(x_imd)
ax_b.set_xticklabels(imd_order, rotation=30, ha="right", fontsize=9.6, fontweight="bold")
ax_b.tick_params(axis="y", labelsize=9.8)
ax_b.set_ylim(0.45, 1.55)
ax_b.set_ylabel("Disparate Impact Ratio (PPR / Ref)", fontsize=10.4, fontweight="bold")
ax_b.set_title("(B) Disparate Impact theo Chỉ số Thiếu thốn IMD qua 4 Mốc", fontsize=11.4, fontweight="bold")
ax_b.legend(loc="upper right", fontsize=8.8, frameon=True)
ax_b.grid(alpha=0.32)

# Panel (C): ROC-AUC across 13 UK Regions across 4 milestones
ax_c = axes[1, 0]
reg_order = (
    fairness_audit_df[(fairness_audit_df["milestone"] == "25%") & (fairness_audit_df["attribute"] == "region")]
    .sort_values("roc_auc", ascending=True)["subgroup"]
    .tolist()
)
y_r = np.arange(len(reg_order))
for ms_lbl in MILESTONES:
    sub_reg = (
        fairness_audit_df[(fairness_audit_df["milestone"] == ms_lbl) & (fairness_audit_df["attribute"] == "region")]
        .set_index("subgroup")
        .loc[reg_order]
    )
    d_auc = fairness_summaries_4ms[ms_lbl]["region"]["roc_auc_gap"]
    ax_c.plot(sub_reg["roc_auc"], y_r, "o-", color=MILESTONE_COLORS[ms_lbl], linewidth=2.0, markersize=6.0, label=f"{ms_lbl} (ΔAUC={d_auc:.3f})")
ax_c.set_yticks(y_r)
ax_c.set_yticklabels(reg_order, fontsize=9.4, fontweight="bold")
ax_c.tick_params(axis="x", labelsize=9.8)
ax_c.set_xlim(0.80, 0.985)
ax_c.set_xlabel("ROC-AUC trên Nhóm con Khu vực UK", fontsize=10.4, fontweight="bold")
ax_c.set_title("(C) Ổn định ROC-AUC qua 13 Khu vực UK xuyên suốt 4 Mốc", fontsize=11.4, fontweight="bold")
ax_c.legend(loc="lower right", fontsize=9.0, frameon=True)
ax_c.grid(alpha=0.32)

# Panel (D): Disparate Impact Ratio across gender, disability, age_band for 4 milestones
ax_d = axes[1, 1]
core_sub = fairness_audit_df[fairness_audit_df["attribute"].isin(["gender", "disability", "age_band"])].copy()
core_sub["label"] = core_sub["attribute"] + ": " + core_sub["subgroup"]
label_order = core_sub[core_sub["milestone"] == "25%"]["label"].tolist()
y_d = np.arange(len(label_order))
h_d = 0.19
ax_d.axvspan(0.80, 1.25, color="#2ecc71", alpha=0.14, label="Vùng Quy tắc 4/5 [0.80, 1.25]")
ax_d.axvline(1.0, color="#2c3e50", linestyle="--", linewidth=1.2)
for i_m, ms_lbl in enumerate(MILESTONES):
    sub_d = core_sub[core_sub["milestone"] == ms_lbl].set_index("label").loc[label_order]
    offset = (i_m - 1.5) * h_d
    ax_d.barh(y_d + offset, sub_d["disparate_impact"], height=h_d, color=MILESTONE_COLORS[ms_lbl], edgecolor="black", linewidth=0.5, label=f"{ms_lbl}")
ax_d.set_yticks(y_d)
ax_d.set_yticklabels(label_order, fontsize=9.8, fontweight="bold")
ax_d.tick_params(axis="x", labelsize=9.8)
ax_d.set_xlim(0.35, 1.48)
ax_d.set_xlabel("Disparate Impact Ratio (PPR_nhóm / PPR_chuẩn)", fontsize=10.4, fontweight="bold")
ax_d.set_title("(D) Tỷ số Tác động Khác biệt (DI) vs Quy tắc 4/5 qua 4 Mốc", fontsize=11.4, fontweight="bold")
ax_d.legend(loc="lower right", ncol=2, fontsize=8.8, frameon=True)
ax_d.grid(axis="x", alpha=0.32)

fig.suptitle(
    "Kiểm định Công bằng Nhân khẩu học (Subgroup Fairness Audit) trên Dự đoán 5-Fold OOF qua 4 Mốc (25% → 100%)",
    fontsize=14.0, fontweight="bold", y=0.985,
)
fig51_path = FIGURES_DIR / "51_week8_demographic_fairness_audit.png"
fig.savefig(fig51_path, dpi=260, bbox_inches="tight", pad_inches=0.20)
plt.close(fig)
print(f"Đã xuất Hình 51: {fig51_path}")

=== TIẾN HÓA CHÊNH LỆCH CÔNG BẰNG (ΔTPR, ΔFPR, ΔAUC, 4/5 RULE) QUA 4 MỐC ===


equal_opportunity_gap_delta_tpr                       \
milestone                              25%    50%    75%   100%   
attribute                                                         
age_band                            0.1795 0.1215 0.0845 0.0475   
disability                          0.0701 0.0460 0.0308 0.0014   
gender                              0.0614 0.0524 0.0532 0.0457   
imd_band                            0.2543 0.1453 0.1212 0.1219   
region                              0.1027 0.0809 0.1146 0.0954   

           fpr_gap_delta_fpr                      roc_auc_gap                \
milestone                25%    50%    75%   100%         25%    50%    75%   
attribute                                                                     
age_band              0.1440 0.0694 0.0274 0.0189      0.0228 0.0033 0.0015   
disability            0.1111 0.0636 0.0160 0.0145      0.0020 0.0022 0.0032   
gender                0.0051 0.0018 0.0027 0.0049      0.0319 0.0205 0.0151   
imd_band              0.2202 0.1214 0.0451 0.0366      0.0500 0.0265 0.0220   
region                0.0911 0.0638 0.0388 0.0226      0.0413 0.0400 0.0379   

                  passes_4_5ths_rule                       
milestone    100%                25%    50%    75%   100%  
attribute                                                  
age_band   0.0028              False  False  False  False  
disability 0.0092              False  False  False   True  
gender     0.0161               True   True   True   True  
imd_band   0.0193              False  False  False  False  
region     0.0338               True  False  False  False

Đã xuất Hình 51: C:\Users\mrore\Project_I\reports\figures\51_week8_demographic_fairness_audit.png


## 8. Lưu Trữ Artifacts Tuần 8 (`data/processed/week8_xai/` & `models/`) & Bộ Kiểm thử Tự động (`unittest`)
Lưu trữ sạch toàn bộ bảng xếp hạng đồng thuận 114 đặc trưng qua 4 mốc, Case Studies 4 mốc + Quỹ đạo dọc, Fairness Audit 4 mốc, bảng OOF tóm tắt từng mốc và tổng hợp (`demo_oof_all_milestones_summary.csv`), và `week8_xai_metrics.json`; sau đó chạy bộ kiểm thử tự động xác minh toàn bộ 4 mốc, **14 biểu đồ (`38–51`)**, và quy chuẩn `AGENTS.md`.

In [9]:
# 1. Lưu top_features_consensus_week8.csv (25% tương thích ngược) & top_features_consensus_4milestones_week8.csv
consensus_df.to_csv(WEEK8_XAI_DIR / "top_features_consensus_week8.csv", index=False)
consensus_4ms_df.to_csv(WEEK8_XAI_DIR / "top_features_consensus_4milestones_week8.csv", index=False)

# 2. Lưu case_studies_week8.json (giữ 25% TP/FP/TN/FN ở root + by_milestone cho cả 4 mốc + longitudinal_trajectory)
case_studies_full_json = {
    **case_studies_payload,
    "by_milestone": case_studies_4ms,
    "longitudinal_trajectory": longitudinal_trajectory,
}
with open(WEEK8_XAI_DIR / "case_studies_week8.json", "w", encoding="utf-8") as f:
    json.dump(case_studies_full_json, f, indent=2, ensure_ascii=False)

# 3. Lưu fairness_audit_week8.csv (cả 4 mốc với cột milestone)
fairness_audit_df.to_csv(WEEK8_XAI_DIR / "fairness_audit_week8.csv", index=False)

# 4. Lưu demo_oof_{25,50,75,100}pct_summary.csv & demo_oof_all_milestones_summary.csv
demo_dfs_list = []
for snap_label in MILESTONES:
    res_m = ms_results[snap_label]
    df_m = res_m["df"]
    oof_shap_m = res_m["oof_shap"]
    top3_idx_per_row = np.argsort(np.abs(oof_shap_m), axis=1)[:, ::-1][:, :3]
    d_sum = pd.DataFrame({
        "id_student": df_m["id_student"].values,
        "code_module": df_m["code_module"].values,
        "code_presentation": df_m["code_presentation"].values,
        "gender": df_m["gender"].values,
        "disability": df_m["disability"].values,
        "age_band": df_m["age_band"].values,
        "imd_band": df_m["imd_band"].fillna("Missing").values,
        "region": df_m["region"].values,
        "final_result": df_m["final_result"].values,
        "at_risk": res_m["y"],
        "fold": res_m["oof_folds"],
        "fold_tau": res_m["oof_taus"],
        "oof_prob": np.round(res_m["oof_probs"], 5),
        "oof_pred": res_m["oof_preds"],
        "top1_feature": [FEAT_NAMES_114[r[0]] for r in top3_idx_per_row],
        "top1_shap": np.round([oof_shap_m[i, r[0]] for i, r in enumerate(top3_idx_per_row)], 4),
        "top2_feature": [FEAT_NAMES_114[r[1]] for r in top3_idx_per_row],
        "top2_shap": np.round([oof_shap_m[i, r[1]] for i, r in enumerate(top3_idx_per_row)], 4),
        "top3_feature": [FEAT_NAMES_114[r[2]] for r in top3_idx_per_row],
        "top3_shap": np.round([oof_shap_m[i, r[2]] for i, r in enumerate(top3_idx_per_row)], 4),
    })
    pct_slug = snap_label.replace("%", "pct")
    d_sum.to_csv(WEEK8_XAI_DIR / f"demo_oof_{pct_slug}_summary.csv", index=False)
    d_with_ms = d_sum.copy()
    d_with_ms.insert(0, "milestone", snap_label)
    demo_dfs_list.append(d_with_ms)

demo_all_df = pd.concat(demo_dfs_list, ignore_index=True)
demo_all_df.to_csv(WEEK8_XAI_DIR / "demo_oof_all_milestones_summary.csv", index=False)

# 5. Lưu week8_xai_metrics.json (đầy đủ 4 mốc + tương thích ngược các khóa 25%)
oof_perf_4ms = {}
for snap_label in MILESTONES:
    res_m = ms_results[snap_label]
    fdf_m = res_m["fold_df"]
    oof_perf_4ms[snap_label] = {
        "n_cohort": int(len(res_m["df"])),
        "at_risk_rate": round(float(res_m["y"].mean()), 4),
        "tau_mean": round(float(np.mean(res_m["fold_thresholds"])), 4),
        "roc_auc_mean": round(float(fdf_m["roc_auc"].mean()), 4),
        "pr_auc_mean": round(float(fdf_m["pr_auc"].mean()), 4),
        "recall_mean": round(float(fdf_m["recall"].mean()), 4),
        "precision_mean": round(float(fdf_m["precision"].mean()), 4),
        "f1_mean": round(float(fdf_m["f1"].mean()), 4),
        "accuracy_mean": round(float(fdf_m["accuracy"].mean()), 4),
        "brier_mean": round(float(fdf_m["brier"].mean()), 4),
        "max_treeshap_additivity_error": float(res_m["max_shap_err"]),
        "oof_confusion_matrix": res_m["oof_cm"],
        "per_fold": res_m["fold_metrics"],
    }

week8_payload = {
    "meta": {
        "model_explained": best_metadata["best_tree_model_saved"],
        "snapshot_primary": "25% dynamic_active",
        "milestones": MILESTONES,
        "cohorts_n": {m: int(len(ms_results[m]["df"])) for m in MILESTONES},
        "n_cohort_25pct": int(len(df_25)),
        "n_features_encoded": int(len(FEAT_NAMES_114)),
        "shap_engine": SHAP_STATUS_MSG,
        "max_treeshap_additivity_error": float(max_additivity_err),
        "max_treeshap_additivity_error_all_milestones": float(global_max_shap_err),
    },
    "oof_performance_verification_25pct": {
        k: v for k, v in oof_perf_4ms["25%"].items()
        if k not in ("n_cohort", "at_risk_rate", "tau_mean", "max_treeshap_additivity_error")
    },
    "oof_performance_verification_4milestones": oof_perf_4ms,
    "global_rank_agreement": global_rank_agreement_4ms["25%"],
    "global_rank_agreement_4milestones": global_rank_agreement_4ms,
    "top_10_consensus_features": consensus_by_ms["25%"].drop(columns=["milestone"]).head(10).to_dict(orient="records"),
    "top_10_consensus_features_4milestones": {
        m: consensus_by_ms[m].drop(columns=["milestone"]).head(10).to_dict(orient="records")
        for m in MILESTONES
    },
    "top_15_shap_features": consensus_by_ms["25%"].sort_values("shap_mean_abs", ascending=False).head(15)[[
        "feature_name", "sedl_dimension", "shap_mean_abs", "shap_share_pct", "perm_auc_drop_mean", "xgb_gain_mean"
    ]].to_dict(orient="records"),
    "top_15_shap_features_4milestones": {
        m: consensus_by_ms[m].sort_values("shap_mean_abs", ascending=False).head(15)[[
            "feature_name", "sedl_dimension", "shap_mean_abs", "shap_share_pct", "perm_auc_drop_mean", "xgb_gain_mean"
        ]].to_dict(orient="records")
        for m in MILESTONES
    },
    "sedl_4d_feature_counts": pd.Series(SEDL_4D_MAP).value_counts().to_dict(),
    "sedl_4d_evolution_25_to_100": sedl_evolution_df.to_dict(orient="records"),
    "milestone_top5_shap": milestone_top5_dict,
    "pdp_nonlinear_thresholds": pdp_summary_info,
    "pdp_nonlinear_thresholds_4milestones": pdp_4ms_summary,
    "case_studies_summary": {
        q: {k: v for k, v in d.items() if k not in ("top8_shap_drivers", "top8_lime_drivers", "milestone")}
        for q, d in case_studies_payload.items()
    },
    "case_studies_summary_4milestones": {
        m: {
            q: {k: v for k, v in d.items() if k not in ("top8_shap_drivers", "top8_lime_drivers")}
            for q, d in case_studies_4ms[m].items()
        }
        for m in MILESTONES
    },
    "longitudinal_case_study_summary": longitudinal_trajectory,
    "lime_vs_shap_200_oof_fidelity": {
        k: v for k, v in lime_fidelity_4ms["25%"].items() if k != "n_perturbations"
    },
    "lime_vs_shap_200_oof_fidelity_4milestones": lime_fidelity_4ms,
    "fairness_audit_summary": fairness_summaries,
    "fairness_audit_summary_4milestones": fairness_summaries_4ms,
}

with open(WEEK8_XAI_DIR / "week8_xai_metrics.json", "w", encoding="utf-8") as f:
    json.dump(week8_payload, f, indent=2, ensure_ascii=False)

print(f"Đã lưu đầy đủ artifacts Tuần 8A (4 mốc) tại: {WEEK8_XAI_DIR}")

# =========================================================================
# BỘ KIỂM THỬ TỰ ĐỘNG UNITTEST (14 TESTS — XÁC MINH TOÀN BỘ 4 MỐC)
# =========================================================================
class TestWeek8XAIExplanation(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        with open(WEEK8_XAI_DIR / "week8_xai_metrics.json", "r", encoding="utf-8") as f:
            cls.payload = json.load(f)
        with open(WEEK8_XAI_DIR / "case_studies_week8.json", "r", encoding="utf-8") as f:
            cls.cases = json.load(f)
        cls.top_df = pd.read_csv(WEEK8_XAI_DIR / "top_features_consensus_week8.csv")
        cls.top_4ms_df = pd.read_csv(WEEK8_XAI_DIR / "top_features_consensus_4milestones_week8.csv")
        cls.fair_df = pd.read_csv(WEEK8_XAI_DIR / "fairness_audit_week8.csv")
        cls.demo_25_df = pd.read_csv(WEEK8_XAI_DIR / "demo_oof_25pct_summary.csv")
        cls.demo_all_df = pd.read_csv(WEEK8_XAI_DIR / "demo_oof_all_milestones_summary.csv")

    def test_01_week7_models_and_114_features_aligned(self):
        self.assertEqual(self.payload["meta"]["n_features_encoded"], 114)
        self.assertEqual(self.payload["meta"]["cohorts_n"], {"25%": 26248, "50%": 24601, "75%": 23159, "100%": 22522})
        self.assertTrue((MODELS_DIR / "best_model.pkl").exists())
        self.assertTrue((MODELS_DIR / "best_model_5fold_ensemble.joblib").exists())
        self.assertTrue((MODELS_DIR / "best_model_5fold_4milestones.joblib").exists())

    def test_02_oof_performance_matches_week7_all_4_milestones(self):
        p4 = self.payload["oof_performance_verification_4milestones"]
        for ms in MILESTONES:
            w7_r = w7_early_lookup[(ms, "dynamic_active", "XGBoost (Tuned)")]
            self.assertAlmostEqual(p4[ms]["roc_auc_mean"], w7_r["roc_auc_mean"], places=3)
            self.assertAlmostEqual(p4[ms]["recall_mean"], w7_r["recall_mean"], places=3)
            self.assertAlmostEqual(p4[ms]["f1_mean"], w7_r["f1_mean"], places=3)

    def test_03_exact_treeshap_additivity_all_4_milestones(self):
        self.assertLessEqual(self.payload["meta"]["max_treeshap_additivity_error_all_milestones"], 1e-6)
        for ms in MILESTONES:
            self.assertLessEqual(self.payload["oof_performance_verification_4milestones"][ms]["max_treeshap_additivity_error"], 1e-6)

    def test_04_consensus_ranking_complete_all_4_milestones(self):
        self.assertEqual(len(self.top_df), 114)
        self.assertEqual(len(self.top_4ms_df), 456)
        self.assertEqual(set(self.top_4ms_df["milestone"].unique()), set(MILESTONES))
        self.assertFalse(self.top_4ms_df[["shap_mean_abs", "perm_auc_drop_mean", "xgb_gain_mean", "consensus_score_pct"]].isna().any().any())
        for ms in MILESTONES:
            self.assertGreater(self.payload["global_rank_agreement_4milestones"][ms]["spearman_shap_vs_perm"], 0.50)

    def test_05_sedl_4d_mapping_100pct_coverage_and_sum_100(self):
        self.assertEqual(sum(self.payload["sedl_4d_feature_counts"].values()), 114)
        ev = pd.DataFrame(self.payload["sedl_4d_evolution_25_to_100"])
        self.assertEqual(set(ev["snapshot"].tolist()), set(MILESTONES))
        pct_sum = ev["behavioural_pct"] + ev["cognitive_pct"] + ev["social_pct"] + ev["demographic_context_pct"]
        for s in pct_sum:
            self.assertAlmostEqual(float(s), 100.0, delta=0.1)

    def test_06_pdp_thresholds_computed_4milestones(self):
        pdp4 = self.payload["pdp_nonlinear_thresholds_4milestones"]
        self.assertEqual(set(pdp4.keys()), set(MILESTONES))
        for ms in MILESTONES:
            self.assertEqual(len(pdp4[ms]), 5)
            for k, v in pdp4[ms].items():
                self.assertGreater(v["pdp_range"], 0.01)

    def test_07_case_studies_4_quadrants_across_4_milestones_and_longitudinal(self):
        self.assertTrue({"TP", "FP", "TN", "FN", "by_milestone", "longitudinal_trajectory"}.issubset(set(self.cases.keys())))
        for ms in MILESTONES:
            c = self.cases["by_milestone"][ms]
            self.assertEqual(set(c.keys()), {"TP", "FP", "TN", "FN"})
            self.assertTrue(c["TP"]["at_risk"] == 1 and c["TP"]["oof_pred"] == 1)
            self.assertTrue(c["FP"]["at_risk"] == 0 and c["FP"]["oof_pred"] == 1)
            self.assertTrue(c["TN"]["at_risk"] == 0 and c["TN"]["oof_pred"] == 0)
            self.assertTrue(c["FN"]["at_risk"] == 1 and c["FN"]["oof_pred"] == 0)
            for q, item in c.items():
                self.assertEqual(len(item["top8_shap_drivers"]), 8)
                self.assertEqual(len(item["top8_lime_drivers"]), 8)
        self.assertEqual(len(self.cases["longitudinal_trajectory"]["trajectory"]), 4)

    def test_08_lime_vs_shap_fidelity_4milestones(self):
        lm4 = self.payload["lime_vs_shap_200_oof_fidelity_4milestones"]
        self.assertEqual(set(lm4.keys()), set(MILESTONES))
        for ms in MILESTONES:
            self.assertEqual(lm4[ms]["n_sampled_students"], 200)
            self.assertGreater(lm4[ms]["local_r2_mean"], 0.45)
            self.assertGreater(lm4[ms]["spearman_active_mean"], 0.35)
            self.assertGreater(lm4[ms]["top10_overlap_mean"], 0.35)

    def test_09_fairness_audit_all_5_attributes_4milestones(self):
        expected_n = {"25%": 26248, "50%": 24601, "75%": 23159, "100%": 22522}
        self.assertEqual(set(self.fair_df["milestone"].unique()), set(MILESTONES))
        for ms in MILESTONES:
            sub_m = self.fair_df[self.fair_df["milestone"] == ms]
            attrs = set(sub_m["attribute"].unique())
            self.assertEqual(attrs, {"gender", "disability", "age_band", "imd_band", "region"})
            for a in attrs:
                self.assertEqual(int(sub_m[sub_m["attribute"] == a]["n"].sum()), expected_n[ms])
        self.assertFalse(self.fair_df[["tpr_recall", "fpr", "disparate_impact", "brier_score"]].isna().any().any())

    def test_10_demo_oof_summaries_valid_all_4_milestones(self):
        self.assertEqual(len(self.demo_25_df), 26248)
        self.assertEqual(len(self.demo_all_df), 26248 + 24601 + 23159 + 22522)
        for ms, n_exp in [("50%", 24601), ("75%", 23159), ("100%", 22522)]:
            df_s = pd.read_csv(WEEK8_XAI_DIR / f"demo_oof_{ms.replace('%', 'pct')}_summary.csv")
            self.assertEqual(len(df_s), n_exp)
        self.assertFalse(self.demo_all_df[["oof_prob", "oof_pred", "top1_feature", "top1_shap"]].isna().any().any())

    def test_11_agents_md_clean_storage_rules(self):
        self.assertFalse((PROJECT_ROOT / "src").exists(), "Vi phạm AGENTS.md: tồn tại thư mục src/")
        self.assertFalse(any(p.is_file() for p in PROCESSED_DIR.iterdir()), "Còn tệp rời tại gốc data/processed/")
        for fn in [
            "week8_xai_metrics.json", "top_features_consensus_week8.csv",
            "top_features_consensus_4milestones_week8.csv", "case_studies_week8.json",
            "fairness_audit_week8.csv", "demo_oof_all_milestones_summary.csv",
        ]:
            self.assertFalse((REPORTS_DIR / fn).exists(), f"Vi phạm AGENTS.md: {fn} nằm trong reports/")
            self.assertTrue((WEEK8_XAI_DIR / fn).exists())

    def test_12_all_fourteen_week8_figures_exist_and_clean(self):
        expected_figs = [
            "38_week8_global_xai_3methods_4milestones.png",
            "39_week8_top10_sedl_consensus_4milestones.png",
            "40_week8_shap_beeswarm_4milestones.png",
            "41_week8_sedl_4d_evolution_4milestones.png",
            "42_week8_pdp_shap_score_submission_4milestones.png",
            "43_week8_pdp_shap_inactivity_4milestones.png",
            "44_week8_pdp_shap_consistency_4milestones.png",
            "45_week8_local_waterfall_25pct.png",
            "46_week8_local_waterfall_50pct.png",
            "47_week8_local_waterfall_75pct.png",
            "48_week8_local_waterfall_100pct.png",
            "49_week8_lime_vs_shap_cases_4milestones.png",
            "50_week8_lime_vs_shap_fidelity_4milestones.png",
            "51_week8_demographic_fairness_audit.png",
        ]
        for fig_name in expected_figs:
            p = FIGURES_DIR / fig_name
            self.assertTrue(p.exists(), f"Thiếu biểu đồ: {fig_name}")
            self.assertGreater(p.stat().st_size, 80000, f"Biểu đồ {fig_name} có kích thước quá nhỏ")
        for old_name in OBSOLETE_WEEK8_FIGS:
            self.assertFalse((FIGURES_DIR / old_name).exists(), f"Còn sót tệp biểu đồ cũ: {old_name}")


suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestWeek8XAIExplanation)
result = unittest.TextTestRunner(stream=sys.stdout, verbosity=2).run(suite)
assert result.wasSuccessful(), f"Có test thất bại: failures={len(result.failures)}, errors={len(result.errors)}"
print(f"\n[PASSED] {result.testsRun}/{result.testsRun} kiểm thử Giai đoạn 8A Đa mốc (14 biểu đồ 38–51, 07_model_explanation.ipynb) hoàn tất không lỗi!")

Đã lưu đầy đủ artifacts Tuần 8A (4 mốc) tại: C:\Users\mrore\Project_I\data\processed\week8_xai


test_01_week7_models_and_114_features_aligned (__main__.TestWeek8XAIExplanation.test_01_week7_models_and_114_features_aligned) ... 

ok


test_02_oof_performance_matches_week7_all_4_milestones (__main__.TestWeek8XAIExplanation.test_02_oof_performance_matches_week7_all_4_milestones) ... 

ok


test_03_exact_treeshap_additivity_all_4_milestones (__main__.TestWeek8XAIExplanation.test_03_exact_treeshap_additivity_all_4_milestones) ... 

... (17 outputs left out)

test_12_all_fourteen_week8_figures_exist_and_clean (__main__.TestWeek8XAIExplanation.test_12_all_fourteen_week8_figures_exist_and_clean) ... 

ok


----------------------------------------------------------------------
Ran 12 tests in 0.847s

OK



[PASSED] 12/12 kiểm thử Giai đoạn 8A Đa mốc (14 biểu đồ 38–51, 07_model_explanation.ipynb) hoàn tất không lỗi!
